# Limestone Data Challenge — Alt Approach Submission

7 Farmers / 46 Indices classification. All trading functions use streaming row accumulation
for temporal lookback (no training CSV dependency at inference time).

**Farmers**: col_52, col_24, col_31, col_34, col_32, col_12, col_49
**Indices**: remaining 46 columns

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('limestone_data_challenge_2026.data.csv')
cols = [c for c in df.columns if c != 'time']
print(f"Loaded: {df.shape[0]} rows x {len(cols)} columns, NaN frac: {df[cols].isna().mean().mean():.4f}")

Loaded: 3650 rows x 53 columns, NaN frac: 0.4883


## Problem 1: 7 Farmers / 46 Indices

Identified via residual variance after regressing on market factor. Each of the 46 indices
is a convex combination of the 7 farmers, with weights found via FFT candidate selection + NNLS.

In [3]:
def solution() -> pd.DataFrame:
    INDEX_COEFS = {
    'col_00': {'col_31': 0.2982, 'col_32': 0.2267, 'col_49': 0.2142, 'col_24': 0.1162, 'col_12': 0.086, 'col_34': 0.0435, 'col_52': 0.0037},
    'col_01': {'col_49': 0.2973, 'col_34': 0.2019, 'col_32': 0.1771, 'col_31': 0.1735, 'col_12': 0.158},
    'col_02': {'col_32': 0.2852, 'col_12': 0.2305, 'col_24': 0.1598, 'col_31': 0.1513, 'col_49': 0.0947, 'col_34': 0.0507, 'col_52': 0.0262},
    'col_03': {'col_49': 0.2697, 'col_32': 0.2261, 'col_12': 0.165, 'col_31': 0.1629, 'col_34': 0.1305, 'col_24': 0.0482},
    'col_04': {'col_49': 0.4044, 'col_34': 0.1861, 'col_32': 0.1225, 'col_12': 0.1094, 'col_31': 0.1006, 'col_24': 0.0923},
    'col_05': {'col_49': 0.29, 'col_12': 0.2476, 'col_52': 0.1806, 'col_32': 0.1448, 'col_31': 0.1412, 'col_34': 0.008},
    'col_06': {'col_12': 0.3848, 'col_31': 0.2594, 'col_32': 0.2371, 'col_24': 0.0625, 'col_49': 0.0478},
    'col_07': {'col_12': 0.3667, 'col_32': 0.2035, 'col_31': 0.1917, 'col_49': 0.1447, 'col_24': 0.0788, 'col_34': 0.0049},
    'col_08': {'col_12': 0.2211, 'col_49': 0.199, 'col_31': 0.1885, 'col_34': 0.1837, 'col_24': 0.1407, 'col_32': 0.0677},
    'col_09': {'col_24': 0.2902, 'col_32': 0.2425, 'col_31': 0.1771, 'col_49': 0.1402, 'col_52': 0.1339, 'col_34': 0.009, 'col_12': 0.0054},
    'col_10': {'col_49': 0.3891, 'col_34': 0.2248, 'col_12': 0.167, 'col_31': 0.1167, 'col_32': 0.1162},
    'col_11': {'col_49': 0.238, 'col_32': 0.2152, 'col_12': 0.1582, 'col_34': 0.148, 'col_24': 0.1187, 'col_31': 0.0962, 'col_52': 0.0202},
    'col_13': {'col_49': 0.2991, 'col_12': 0.2239, 'col_31': 0.1601, 'col_32': 0.1202, 'col_34': 0.1029, 'col_24': 0.0934},
    'col_14': {'col_12': 0.358, 'col_49': 0.2429, 'col_31': 0.1985, 'col_32': 0.1091, 'col_24': 0.0445, 'col_52': 0.0364},
    'col_15': {'col_49': 0.2744, 'col_12': 0.2381, 'col_32': 0.2147, 'col_34': 0.1121, 'col_31': 0.0816, 'col_52': 0.0716},
    'col_16': {'col_31': 0.2571, 'col_49': 0.2513, 'col_24': 0.1888, 'col_32': 0.1688, 'col_12': 0.0964, 'col_34': 0.0339},
    'col_17': {'col_49': 0.2366, 'col_32': 0.2011, 'col_12': 0.1723, 'col_24': 0.1586, 'col_31': 0.1546, 'col_34': 0.0694, 'col_52': 0.0073},
    'col_18': {'col_49': 0.2446, 'col_34': 0.1865, 'col_32': 0.1631, 'col_12': 0.1464, 'col_31': 0.1351, 'col_24': 0.1237},
    'col_19': {'col_49': 0.3245, 'col_32': 0.208, 'col_31': 0.194, 'col_12': 0.1641, 'col_34': 0.1102},
    'col_20': {'col_32': 0.2502, 'col_49': 0.1779, 'col_34': 0.1706, 'col_24': 0.1693, 'col_12': 0.1157, 'col_31': 0.0922, 'col_52': 0.025},
    'col_21': {'col_32': 0.2124, 'col_34': 0.1953, 'col_12': 0.1738, 'col_31': 0.1668, 'col_49': 0.1316, 'col_24': 0.1214},
    'col_22': {'col_49': 0.373, 'col_24': 0.1974, 'col_32': 0.1869, 'col_31': 0.1277, 'col_34': 0.0655, 'col_52': 0.0399, 'col_12': 0.0145},
    'col_23': {'col_32': 0.4033, 'col_24': 0.2019, 'col_12': 0.1443, 'col_31': 0.0972, 'col_52': 0.0887, 'col_34': 0.0623, 'col_49': 0.0048},
    'col_25': {'col_32': 0.2682, 'col_49': 0.2546, 'col_31': 0.2417, 'col_34': 0.1158, 'col_24': 0.0909, 'col_52': 0.0341},
    'col_26': {'col_49': 0.2693, 'col_31': 0.1938, 'col_34': 0.1771, 'col_24': 0.1552, 'col_32': 0.1119, 'col_12': 0.0496, 'col_52': 0.0404},
    'col_27': {'col_49': 0.3749, 'col_31': 0.1866, 'col_12': 0.1775, 'col_32': 0.1192, 'col_34': 0.0794, 'col_24': 0.0651},
    'col_28': {'col_49': 0.3615, 'col_12': 0.178, 'col_34': 0.1212, 'col_32': 0.1116, 'col_52': 0.0947, 'col_24': 0.0715, 'col_31': 0.0565},
    'col_29': {'col_12': 0.2496, 'col_49': 0.2268, 'col_32': 0.1854, 'col_31': 0.1193, 'col_24': 0.1067, 'col_34': 0.1048},
    'col_30': {'col_34': 0.2342, 'col_49': 0.2174, 'col_31': 0.164, 'col_24': 0.1409, 'col_32': 0.1179, 'col_12': 0.114, 'col_52': 0.0117},
    'col_33': {'col_49': 0.2641, 'col_12': 0.2294, 'col_31': 0.197, 'col_32': 0.1608, 'col_34': 0.133, 'col_24': 0.0163},
    'col_35': {'col_49': 0.4331, 'col_31': 0.1498, 'col_52': 0.1368, 'col_32': 0.1079, 'col_34': 0.0961, 'col_12': 0.0885},
    'col_36': {'col_49': 0.2698, 'col_32': 0.2323, 'col_12': 0.2249, 'col_31': 0.1382, 'col_34': 0.137},
    'col_37': {'col_24': 0.1881, 'col_31': 0.1748, 'col_12': 0.1658, 'col_32': 0.1655, 'col_49': 0.1548, 'col_52': 0.1166, 'col_34': 0.0258},
    'col_38': {'col_49': 0.3553, 'col_12': 0.1793, 'col_32': 0.1762, 'col_34': 0.1002, 'col_31': 0.0839, 'col_52': 0.0608, 'col_24': 0.0379},
    'col_39': {'col_12': 0.2455, 'col_31': 0.196, 'col_32': 0.1899, 'col_49': 0.1844, 'col_24': 0.1294, 'col_34': 0.0635},
    'col_40': {'col_49': 0.2821, 'col_31': 0.233, 'col_34': 0.1454, 'col_24': 0.1219, 'col_32': 0.0962, 'col_12': 0.0691, 'col_52': 0.0505},
    'col_41': {'col_49': 0.3123, 'col_32': 0.2347, 'col_31': 0.1579, 'col_34': 0.1513, 'col_12': 0.1366},
    'col_42': {'col_32': 0.2226, 'col_49': 0.206, 'col_31': 0.1972, 'col_34': 0.1212, 'col_52': 0.1124, 'col_24': 0.1087, 'col_12': 0.0249},
    'col_43': {'col_12': 0.2697, 'col_49': 0.1978, 'col_34': 0.1646, 'col_31': 0.1067, 'col_32': 0.1024, 'col_24': 0.0866, 'col_52': 0.0774},
    'col_44': {'col_49': 0.1978, 'col_12': 0.1892, 'col_31': 0.1846, 'col_24': 0.1775, 'col_32': 0.1599, 'col_52': 0.0533, 'col_34': 0.0331},
    'col_45': {'col_31': 0.231, 'col_34': 0.2205, 'col_12': 0.1656, 'col_24': 0.1414, 'col_32': 0.1357, 'col_49': 0.1081},
    'col_46': {'col_34': 0.3336, 'col_32': 0.2044, 'col_12': 0.1568, 'col_24': 0.1213, 'col_49': 0.0979, 'col_31': 0.0688, 'col_52': 0.0169},
    'col_47': {'col_49': 0.3221, 'col_12': 0.2235, 'col_31': 0.1433, 'col_34': 0.1079, 'col_32': 0.0884, 'col_24': 0.0656, 'col_52': 0.0398},
    'col_48': {'col_12': 0.239, 'col_32': 0.1943, 'col_31': 0.1902, 'col_49': 0.1457, 'col_24': 0.143, 'col_34': 0.052, 'col_52': 0.0288},
    'col_50': {'col_32': 0.4025, 'col_49': 0.1347, 'col_24': 0.1291, 'col_31': 0.1258, 'col_12': 0.1222, 'col_34': 0.0798},
    'col_51': {'col_31': 0.2271, 'col_49': 0.2068, 'col_32': 0.1827, 'col_12': 0.1795, 'col_24': 0.154, 'col_34': 0.0451},
}
    rows = []
    for idx_col, constituents in INDEX_COEFS.items():
        for const_col, coef in constituents.items():
            rows.append((idx_col, const_col, coef))
    return pd.DataFrame(rows, columns=["index_col", "constituent_col", "coef"])

p1_result = solution()
print(f"{len(p1_result)} rows across {p1_result['index_col'].nunique()} indices")
display(p1_result.head(20))

290 rows across 46 indices


,index_col,constituent_col,coef
0,col_00,col_31,0.2982
1,col_00,col_32,0.2267
2,col_00,col_49,0.2142
3,col_00,col_24,0.1162
4,col_00,col_12,0.0860
5,col_00,col_34,0.0435
6,col_00,col_52,0.0037
7,col_01,col_49,0.2973
8,col_01,col_34,0.2019
9,col_01,col_32,0.1771


## Problem 2: Imputation (Alt Approach)

Same pipeline as original (periodic prefill + reverse inference + KNN ensemble) but using
the 46-index formulas with 7-farmer weights for algebraic fills.

In [4]:
def solve_problem_2(
    input_df,
    time_col='time',
    periodic_cols=None,
    periodic_candidate_k_values=(1, 2),
    periodic_min_time_for_fit=30,
    median_fill_cols=None,
    return_periodic_details=False,
):
    """Fill NaNs using the notebook's final Problem 2 pipeline.

    This version is self-contained: all coefficients and helpers live inside the
    function so it can be called independently from a fresh notebook kernel.
    """
    import numpy as np
    import pandas as pd
    from scipy.signal import lombscargle
    from sklearn.impute import KNNImputer

    index_coefs = {
        'col_00': {'col_31': 0.2982, 'col_32': 0.2267, 'col_49': 0.2142, 'col_24': 0.1162, 'col_12': 0.086, 'col_34': 0.0435, 'col_52': 0.0037},
        'col_01': {'col_49': 0.2973, 'col_34': 0.2019, 'col_32': 0.1771, 'col_31': 0.1735, 'col_12': 0.158},
        'col_02': {'col_32': 0.2852, 'col_12': 0.2305, 'col_24': 0.1598, 'col_31': 0.1513, 'col_49': 0.0947, 'col_34': 0.0507, 'col_52': 0.0262},
        'col_03': {'col_49': 0.2697, 'col_32': 0.2261, 'col_12': 0.165, 'col_31': 0.1629, 'col_34': 0.1305, 'col_24': 0.0482},
        'col_04': {'col_49': 0.4044, 'col_34': 0.1861, 'col_32': 0.1225, 'col_12': 0.1094, 'col_31': 0.1006, 'col_24': 0.0923},
        'col_05': {'col_49': 0.29, 'col_12': 0.2476, 'col_52': 0.1806, 'col_32': 0.1448, 'col_31': 0.1412, 'col_34': 0.008},
        'col_06': {'col_12': 0.3848, 'col_31': 0.2594, 'col_32': 0.2371, 'col_24': 0.0625, 'col_49': 0.0478},
        'col_07': {'col_12': 0.3667, 'col_32': 0.2035, 'col_31': 0.1917, 'col_49': 0.1447, 'col_24': 0.0788, 'col_34': 0.0049},
        'col_08': {'col_12': 0.2211, 'col_49': 0.199, 'col_31': 0.1885, 'col_34': 0.1837, 'col_24': 0.1407, 'col_32': 0.0677},
        'col_09': {'col_24': 0.2902, 'col_32': 0.2425, 'col_31': 0.1771, 'col_49': 0.1402, 'col_52': 0.1339, 'col_34': 0.009, 'col_12': 0.0054},
        'col_10': {'col_49': 0.3891, 'col_34': 0.2248, 'col_12': 0.167, 'col_31': 0.1167, 'col_32': 0.1162},
        'col_11': {'col_49': 0.238, 'col_32': 0.2152, 'col_12': 0.1582, 'col_34': 0.148, 'col_24': 0.1187, 'col_31': 0.0962, 'col_52': 0.0202},
        'col_13': {'col_49': 0.2991, 'col_12': 0.2239, 'col_31': 0.1601, 'col_32': 0.1202, 'col_34': 0.1029, 'col_24': 0.0934},
        'col_14': {'col_12': 0.358, 'col_49': 0.2429, 'col_31': 0.1985, 'col_32': 0.1091, 'col_24': 0.0445, 'col_52': 0.0364},
        'col_15': {'col_49': 0.2744, 'col_12': 0.2381, 'col_32': 0.2147, 'col_34': 0.1121, 'col_31': 0.0816, 'col_52': 0.0716},
        'col_16': {'col_31': 0.2571, 'col_49': 0.2513, 'col_24': 0.1888, 'col_32': 0.1688, 'col_12': 0.0964, 'col_34': 0.0339},
        'col_17': {'col_49': 0.2366, 'col_32': 0.2011, 'col_12': 0.1723, 'col_24': 0.1586, 'col_31': 0.1546, 'col_34': 0.0694, 'col_52': 0.0073},
        'col_18': {'col_49': 0.2446, 'col_34': 0.1865, 'col_32': 0.1631, 'col_12': 0.1464, 'col_31': 0.1351, 'col_24': 0.1237},
        'col_19': {'col_49': 0.3245, 'col_32': 0.208, 'col_31': 0.194, 'col_12': 0.1641, 'col_34': 0.1102},
        'col_20': {'col_32': 0.2502, 'col_49': 0.1779, 'col_34': 0.1706, 'col_24': 0.1693, 'col_12': 0.1157, 'col_31': 0.0922, 'col_52': 0.025},
        'col_21': {'col_32': 0.2124, 'col_34': 0.1953, 'col_12': 0.1738, 'col_31': 0.1668, 'col_49': 0.1316, 'col_24': 0.1214},
        'col_22': {'col_49': 0.373, 'col_24': 0.1974, 'col_32': 0.1869, 'col_31': 0.1277, 'col_34': 0.0655, 'col_52': 0.0399, 'col_12': 0.0145},
        'col_23': {'col_32': 0.4033, 'col_24': 0.2019, 'col_12': 0.1443, 'col_31': 0.0972, 'col_52': 0.0887, 'col_34': 0.0623, 'col_49': 0.0048},
        'col_25': {'col_32': 0.2682, 'col_49': 0.2546, 'col_31': 0.2417, 'col_34': 0.1158, 'col_24': 0.0909, 'col_52': 0.0341},
        'col_26': {'col_49': 0.2693, 'col_31': 0.1938, 'col_34': 0.1771, 'col_24': 0.1552, 'col_32': 0.1119, 'col_12': 0.0496, 'col_52': 0.0404},
        'col_27': {'col_49': 0.3749, 'col_31': 0.1866, 'col_12': 0.1775, 'col_32': 0.1192, 'col_34': 0.0794, 'col_24': 0.0651},
        'col_28': {'col_49': 0.3615, 'col_12': 0.178, 'col_34': 0.1212, 'col_32': 0.1116, 'col_52': 0.0947, 'col_24': 0.0715, 'col_31': 0.0565},
        'col_29': {'col_12': 0.2496, 'col_49': 0.2268, 'col_32': 0.1854, 'col_31': 0.1193, 'col_24': 0.1067, 'col_34': 0.1048},
        'col_30': {'col_34': 0.2342, 'col_49': 0.2174, 'col_31': 0.164, 'col_24': 0.1409, 'col_32': 0.1179, 'col_12': 0.114, 'col_52': 0.0117},
        'col_33': {'col_49': 0.2641, 'col_12': 0.2294, 'col_31': 0.197, 'col_32': 0.1608, 'col_34': 0.133, 'col_24': 0.0163},
        'col_35': {'col_49': 0.4331, 'col_31': 0.1498, 'col_52': 0.1368, 'col_32': 0.1079, 'col_34': 0.0961, 'col_12': 0.0885},
        'col_36': {'col_49': 0.2698, 'col_32': 0.2323, 'col_12': 0.2249, 'col_31': 0.1382, 'col_34': 0.137},
        'col_37': {'col_24': 0.1881, 'col_31': 0.1748, 'col_12': 0.1658, 'col_32': 0.1655, 'col_49': 0.1548, 'col_52': 0.1166, 'col_34': 0.0258},
        'col_38': {'col_49': 0.3553, 'col_12': 0.1793, 'col_32': 0.1762, 'col_34': 0.1002, 'col_31': 0.0839, 'col_52': 0.0608, 'col_24': 0.0379},
        'col_39': {'col_12': 0.2455, 'col_31': 0.196, 'col_32': 0.1899, 'col_49': 0.1844, 'col_24': 0.1294, 'col_34': 0.0635},
        'col_40': {'col_49': 0.2821, 'col_31': 0.233, 'col_34': 0.1454, 'col_24': 0.1219, 'col_32': 0.0962, 'col_12': 0.0691, 'col_52': 0.0505},
        'col_41': {'col_49': 0.3123, 'col_32': 0.2347, 'col_31': 0.1579, 'col_34': 0.1513, 'col_12': 0.1366},
        'col_42': {'col_32': 0.2226, 'col_49': 0.206, 'col_31': 0.1972, 'col_34': 0.1212, 'col_52': 0.1124, 'col_24': 0.1087, 'col_12': 0.0249},
        'col_43': {'col_12': 0.2697, 'col_49': 0.1978, 'col_34': 0.1646, 'col_31': 0.1067, 'col_32': 0.1024, 'col_24': 0.0866, 'col_52': 0.0774},
        'col_44': {'col_49': 0.1978, 'col_12': 0.1892, 'col_31': 0.1846, 'col_24': 0.1775, 'col_32': 0.1599, 'col_52': 0.0533, 'col_34': 0.0331},
        'col_45': {'col_31': 0.231, 'col_34': 0.2205, 'col_12': 0.1656, 'col_24': 0.1414, 'col_32': 0.1357, 'col_49': 0.1081},
        'col_46': {'col_34': 0.3336, 'col_32': 0.2044, 'col_12': 0.1568, 'col_24': 0.1213, 'col_49': 0.0979, 'col_31': 0.0688, 'col_52': 0.0169},
        'col_47': {'col_49': 0.3221, 'col_12': 0.2235, 'col_31': 0.1433, 'col_34': 0.1079, 'col_32': 0.0884, 'col_24': 0.0656, 'col_52': 0.0398},
        'col_48': {'col_12': 0.239, 'col_32': 0.1943, 'col_31': 0.1902, 'col_49': 0.1457, 'col_24': 0.143, 'col_34': 0.052, 'col_52': 0.0288},
        'col_50': {'col_32': 0.4025, 'col_49': 0.1347, 'col_24': 0.1291, 'col_31': 0.1258, 'col_12': 0.1222, 'col_34': 0.0798},
        'col_51': {'col_31': 0.2271, 'col_49': 0.2068, 'col_32': 0.1827, 'col_12': 0.1795, 'col_24': 0.154, 'col_34': 0.0451},
    }

    min_coef_for_reverse = 0.05
    long_gap_threshold = 5
    default_periodic_cols = ['col_24', 'col_12', 'col_31', 'col_32', 'col_34', 'col_49']
    default_median_fill_cols = ['col_52', 'col_34']

    if periodic_cols is None:
        periodic_cols = default_periodic_cols
    if median_fill_cols is None:
        median_fill_cols = default_median_fill_cols

    cols = [c for c in input_df.columns if c != time_col]
    n_rows = len(input_df)
    n_cols = len(cols)

    original_data = input_df[cols].to_numpy(dtype=float).copy()
    original_nan_mask = np.isnan(original_data)
    original_nan_total = int(original_nan_mask.sum())

    def select_distinct_frequencies(freqs, power, top_k=3, min_rel_gap=0.08):
        order = np.argsort(power)[::-1]
        chosen = []
        for idx in order:
            freq = float(freqs[idx])
            if freq <= 0:
                continue
            if all(abs(freq - prev) / max(prev, 1e-12) >= min_rel_gap for prev in chosen):
                chosen.append(freq)
            if len(chosen) >= top_k:
                break
        return chosen

    def fit_with_selected_frequencies(time_all, y_all, fit_mask, chosen_freqs):
        t_fit = time_all[fit_mask]
        y_fit = y_all[fit_mask]

        t_center = float(np.mean(t_fit))
        x_fit = t_fit - t_center
        x_all = time_all - t_center

        x_lin_fit = np.column_stack([np.ones(len(x_fit)), x_fit])
        lin_coef, _, _, _ = np.linalg.lstsq(x_lin_fit, y_fit, rcond=None)
        line_fit_fit = x_lin_fit @ lin_coef
        detrended_fit = y_fit - line_fit_fit

        periodic_design_fit = []
        periodic_design_all = []
        for freq in chosen_freqs:
            ang_fit = 2 * np.pi * freq * t_fit
            ang_all = 2 * np.pi * freq * time_all
            periodic_design_fit.extend([np.sin(ang_fit), np.cos(ang_fit)])
            periodic_design_all.extend([np.sin(ang_all), np.cos(ang_all)])

        if periodic_design_fit:
            x_periodic_fit = np.column_stack(periodic_design_fit)
            periodic_coef, _, _, _ = np.linalg.lstsq(x_periodic_fit, detrended_fit, rcond=None)
            periodic_fit_fit = x_periodic_fit @ periodic_coef
            x_periodic_all = np.column_stack(periodic_design_all)
            periodic_fit_all = x_periodic_all @ periodic_coef
        else:
            periodic_fit_fit = np.zeros(len(t_fit), dtype=float)
            periodic_fit_all = np.zeros(len(time_all), dtype=float)

        line_fit_all = lin_coef[0] + lin_coef[1] * x_all
        fitted_fit = line_fit_fit + periodic_fit_fit
        fitted_all = line_fit_all + periodic_fit_all

        resid = y_fit - fitted_fit
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        denom = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
        r2 = float(1 - np.sum(resid ** 2) / denom) if denom > 0 else np.nan

        return {
            'lin_coef': lin_coef,
            'line_fit_all': line_fit_all,
            'periodic_fit_all': periodic_fit_all,
            'fitted_all': fitted_all,
            'rmse': rmse,
            'r2': r2,
        }

    def fit_trend_plus_periodic_column(local_df, col):
        time_all = local_df[time_col].to_numpy(dtype=float)
        y_all = local_df[col].to_numpy(dtype=float)
        obs_mask = ~np.isnan(y_all)
        if obs_mask.sum() < 20:
            raise ValueError(f'Not enough observed values to fit {col}')

        fit_mask = obs_mask & (time_all >= periodic_min_time_for_fit)
        if fit_mask.sum() < 20:
            fit_mask = obs_mask.copy()

        t_fit = time_all[fit_mask]
        y_fit = y_all[fit_mask]
        time_span = float(np.max(t_fit) - np.min(t_fit))
        unique_t = np.unique(np.sort(time_all))
        dt = np.median(np.diff(unique_t)) if len(unique_t) > 1 else 1.0
        min_freq = max(1.0 / max(time_span, 1.0), 1e-4)
        max_freq = 0.5 / max(dt, 1e-6)
        freq_grid = np.linspace(min_freq, max_freq, 2000)
        ang_grid = 2 * np.pi * freq_grid

        t_center_init = float(np.mean(t_fit))
        x_fit_init = t_fit - t_center_init
        x_lin_init = np.column_stack([np.ones(len(x_fit_init)), x_fit_init])
        lin_coef_init, _, _, _ = np.linalg.lstsq(x_lin_init, y_fit, rcond=None)
        detrended_fit_init = y_fit - (x_lin_init @ lin_coef_init)
        centered = detrended_fit_init - np.mean(detrended_fit_init)
        power = lombscargle(t_fit, centered, ang_grid, normalize=True)

        max_k = max(periodic_candidate_k_values) if periodic_candidate_k_values else 0
        ordered_freqs = select_distinct_frequencies(freq_grid, power, top_k=max_k)

        candidate_rows = []
        best_result = None
        best_freqs = None
        for k in periodic_candidate_k_values:
            chosen_freqs = ordered_freqs[:k]
            fit_result = fit_with_selected_frequencies(time_all, y_all, fit_mask, chosen_freqs)
            candidate_rows.append({
                'column': col,
                'k': int(k),
                'rmse_on_fit_rows': fit_result['rmse'],
                'r2_on_fit_rows': fit_result['r2'],
                'dominant_periods': [float(1.0 / f) for f in chosen_freqs],
            })
            if best_result is None or fit_result['rmse'] < best_result['rmse']:
                best_result = fit_result
                best_freqs = chosen_freqs

        fit_df = pd.DataFrame({
            time_col: time_all,
            col: y_all,
            f'{col}_line_fit': best_result['line_fit_all'],
            f'{col}_periodic_fit': best_result['periodic_fit_all'],
            f'{col}_fitted': best_result['fitted_all'],
            f'{col}_prefilled': np.where(np.isnan(y_all), best_result['fitted_all'], y_all),
            f'{col}_was_missing': np.isnan(y_all),
            f'{col}_used_for_fit': fit_mask,
        })
        summary = {
            'column': col,
            'n_obs': int(obs_mask.sum()),
            'n_fit_obs': int(fit_mask.sum()),
            'n_missing': int((~obs_mask).sum()),
            'min_time_for_fit': float(periodic_min_time_for_fit),
            'best_k': int(len(best_freqs)),
            'line_intercept_at_center': float(best_result['lin_coef'][0]),
            'line_slope': float(best_result['lin_coef'][1]),
            'rmse_on_fit_rows': best_result['rmse'],
            'r2_on_fit_rows': best_result['r2'],
            'dominant_periods': [float(1.0 / f) for f in best_freqs],
        }
        candidate_df = pd.DataFrame(candidate_rows).sort_values(['rmse_on_fit_rows', 'k']).reset_index(drop=True)
        return fit_df, summary, candidate_df

    def prefill_periodic_columns(local_df):
        prefilled_df = local_df.copy()
        fit_frames, summaries, candidate_tables = [], [], []
        for col in [c for c in periodic_cols if c in cols]:
            fit_df, summary, candidate_df = fit_trend_plus_periodic_column(prefilled_df, col)
            missing_mask = fit_df[f'{col}_was_missing'].to_numpy()
            prefilled_df.loc[missing_mask, col] = fit_df.loc[missing_mask, f'{col}_fitted'].to_numpy()
            fit_frames.append(fit_df)
            summaries.append(summary)
            candidate_tables.append(candidate_df)
        summary_df = pd.DataFrame(summaries).sort_values('column').reset_index(drop=True) if summaries else pd.DataFrame()
        all_candidates_df = pd.concat(candidate_tables, ignore_index=True) if candidate_tables else pd.DataFrame()
        return prefilled_df, fit_frames, summary_df, all_candidates_df

    def add_temporal_features(td, windows):
        extras = []
        nr, nc = td.shape
        for w in windows:
            extra = np.full((nr, nc), np.nan)
            for i in range(nr):
                lo, hi = max(0, i - w), min(nr, i + w + 1)
                nearby = list(range(lo, i)) + list(range(i + 1, hi))
                if nearby:
                    extra[i] = np.nanmean(td[nearby], axis=0)
            extras.append(extra)
        return np.column_stack([td] + extras)

    periodic_prefill_summary_df = pd.DataFrame()
    periodic_prefill_candidates_df = pd.DataFrame()
    periodic_prefill_fit_frames = []
    working_df = input_df.copy()
    protected_periodic_cols = []
    protected_periodic_mask = np.zeros((n_rows, n_cols), dtype=bool)
    if periodic_cols:
        protected_periodic_cols = [c for c in periodic_cols if c in cols]
        if protected_periodic_cols:
            working_df, periodic_prefill_fit_frames, periodic_prefill_summary_df, periodic_prefill_candidates_df = prefill_periodic_columns(input_df)
            for col in protected_periodic_cols:
                protected_periodic_mask[:, cols.index(col)] = True

    data = working_df[cols].to_numpy(dtype=float).copy()
    nan_mask = np.isnan(data)
    periodic_prefill_count = int(original_nan_total - nan_mask.sum())

    pre_data = data.copy()
    pre_nan = nan_mask.copy()
    pre_adj = 0
    for _ in range(3):
        new_adj = 0
        for idx_col, coefs in index_coefs.items():
            j_idx = cols.index(idx_col)
            farmer_info = [(cols.index(fc), c) for fc, c in coefs.items()]
            for i in range(n_rows):
                if pre_nan[i, j_idx]:
                    continue
                obs_idx_val = pre_data[i, j_idx]
                nan_f = [(j_f, w) for j_f, w in farmer_info if pre_nan[i, j_f]]
                obs_f = [(j_f, w) for j_f, w in farmer_info if not pre_nan[i, j_f]]
                if len(nan_f) != 1:
                    continue
                j_f, w_f = nan_f[0]
                if w_f < min_coef_for_reverse or protected_periodic_mask[i, j_f]:
                    continue
                known_part = sum(w * pre_data[i, jf] for jf, w in obs_f)
                new_val = (obs_idx_val - known_part) / w_f
                if new_val > 0:
                    col_vals = data[:, j_f]
                    local_window = 10
                    lo_w, hi_w = max(0, i - local_window), min(n_rows, i + local_window + 1)
                    nearby_obs = col_vals[lo_w:hi_w]
                    nearby_obs = nearby_obs[~np.isnan(nearby_obs)]
                    if len(nearby_obs) >= 3:
                        local_mean = np.mean(nearby_obs)
                        local_std = max(np.std(nearby_obs), 1e-6)
                        accept = abs(new_val - local_mean) < 2.5 * local_std
                    else:
                        col_mean = np.nanmean(col_vals)
                        col_std = np.nanstd(col_vals)
                        accept = abs(new_val - col_mean) < 2.5 * col_std
                    if accept:
                        pre_data[i, j_f] = new_val
                        pre_nan[i, j_f] = False
                        new_adj += 1
        pre_adj += new_adj
        if new_adj == 0:
            break

    for idx_col, coefs in index_coefs.items():
        j = cols.index(idx_col)
        farmer_info = [(cols.index(fc), c) for fc, c in coefs.items()]
        for i in range(n_rows):
            if not pre_nan[i, j]:
                continue
            if all(not pre_nan[i, jf] for jf, _ in farmer_info):
                val = sum(c * pre_data[i, jf] for jf, c in farmer_info)
                if val > 0 and not protected_periodic_mask[i, j]:
                    pre_data[i, j] = val
                    pre_nan[i, j] = False

    pre_filled_total = int(nan_mask.sum() - pre_nan.sum())

    market_level = np.zeros(n_rows)
    for i in range(n_rows):
        obs_vals = pre_data[i, ~pre_nan[i, :]]
        market_level[i] = np.mean(obs_vals) if len(obs_vals) > 0 else np.nan
    ml_filled = pd.Series(market_level).interpolate(method='linear', limit_direction='both').ffill().bfill().values

    deviations = np.full_like(pre_data, np.nan)
    for i in range(n_rows):
        for j in range(n_cols):
            if not pre_nan[i, j]:
                deviations[i, j] = pre_data[i, j] - ml_filled[i]

    col_mean_dev = np.array([np.nanmean(deviations[:, j]) for j in range(n_cols)])
    gap_matrix = np.zeros((n_rows, n_cols), dtype=int)
    for j in range(n_cols):
        last_obs = -9999
        for i in range(n_rows):
            if not pre_nan[i, j]:
                last_obs = i
                gap_matrix[i, j] = 0
            else:
                gap_matrix[i, j] = i - last_obs if last_obs >= 0 else 9999

    windows = [1, 2, 3, 5, 7]
    combined = add_temporal_features(deviations, windows)
    time_feature = (np.arange(n_rows, dtype=float) / n_rows) * 6.4
    combined = np.column_stack([combined, time_feature.reshape(-1, 1)])

    imps = []
    for k in [2, 3, 5, 7]:
        knn = KNNImputer(n_neighbors=k)
        imps.append(knn.fit_transform(combined)[:, :n_cols])
    dev_imputed = np.mean(imps, axis=0)

    final_imp = dev_imputed + ml_filled[:, None]
    n_overridden = 0
    for i in range(n_rows):
        for j in range(n_cols):
            if pre_nan[i, j] and gap_matrix[i, j] > long_gap_threshold and not protected_periodic_mask[i, j]:
                final_imp[i, j] = ml_filled[i] + col_mean_dev[j]
                n_overridden += 1

    final_imp[~original_nan_mask] = original_data[~original_nan_mask]
    step0_filled = nan_mask & ~pre_nan
    final_imp[step0_filled] = pre_data[step0_filled]
    if protected_periodic_mask.any():
        final_imp[protected_periodic_mask] = data[protected_periodic_mask]

    for idx_col, coefs in index_coefs.items():
        j = cols.index(idx_col)
        val = np.zeros(n_rows)
        for fc, c in coefs.items():
            val += c * final_imp[:, cols.index(fc)]
        for i in range(n_rows):
            if nan_mask[i, j] and not protected_periodic_mask[i, j]:
                final_imp[i, j] = val[i]

    adj = 0
    reverse_inferred = set()
    for idx_col, coefs in index_coefs.items():
        j_idx = cols.index(idx_col)
        farmer_info = [(cols.index(fc), c) for fc, c in coefs.items()]
        for i in range(n_rows):
            if nan_mask[i, j_idx]:
                continue
            obs_idx_val = original_data[i, j_idx]
            imp_farmers = [(j_f, w) for j_f, w in farmer_info if nan_mask[i, j_f]]
            obs_farmers = [(j_f, w) for j_f, w in farmer_info if not nan_mask[i, j_f]]
            if len(imp_farmers) != 1:
                continue
            j_f, w_f = imp_farmers[0]
            if w_f < min_coef_for_reverse or protected_periodic_mask[i, j_f]:
                continue
            known_part = sum(w * original_data[i, jf] for jf, w in obs_farmers)
            new_val = (obs_idx_val - known_part) / w_f
            if new_val > 0:
                col_vals = original_data[:, j_f]
                local_window = 10
                lo_w, hi_w = max(0, i - local_window), min(n_rows, i + local_window + 1)
                nearby_obs = col_vals[lo_w:hi_w]
                nearby_obs = nearby_obs[~np.isnan(nearby_obs)]
                if len(nearby_obs) >= 3:
                    local_mean = np.mean(nearby_obs)
                    local_std = max(np.std(nearby_obs), 1e-6)
                    accept = abs(new_val - local_mean) < 2.5 * local_std
                else:
                    col_mean = np.nanmean(col_vals)
                    col_std = np.nanstd(col_vals)
                    accept = abs(new_val - col_mean) < 2.5 * col_std
                if accept:
                    final_imp[i, j_f] = new_val
                    reverse_inferred.add((i, j_f))
                    adj += 1

    for idx_col, coefs in index_coefs.items():
        j = cols.index(idx_col)
        val = np.zeros(n_rows)
        for fc, c in coefs.items():
            val += c * final_imp[:, cols.index(fc)]
        for i in range(n_rows):
            if nan_mask[i, j] and not protected_periodic_mask[i, j]:
                final_imp[i, j] = val[i]

    effective_median_fill_cols = [c for c in median_fill_cols if c in cols and c not in protected_periodic_cols]
    if effective_median_fill_cols:
        median_col_indices = set(cols.index(mc) for mc in effective_median_fill_cols)
        other_cols_mask = np.array([j not in median_col_indices for j in range(n_cols)])
        for mc in effective_median_fill_cols:
            j_mc = cols.index(mc)
            for i in range(n_rows):
                if not original_nan_mask[i, j_mc]:
                    final_imp[i, j_mc] = original_data[i, j_mc]
                elif (i, j_mc) in reverse_inferred:
                    pass
                elif original_nan_mask[i, j_mc] and not pre_nan[i, j_mc]:
                    final_imp[i, j_mc] = pre_data[i, j_mc]
                else:
                    final_imp[i, j_mc] = np.median(final_imp[i, other_cols_mask])
        for idx_col, coefs in index_coefs.items():
            if not any(fc in effective_median_fill_cols for fc in coefs):
                continue
            j = cols.index(idx_col)
            val = np.zeros(n_rows)
            for fc, c in coefs.items():
                val += c * final_imp[:, cols.index(fc)]
            for i in range(n_rows):
                if original_nan_mask[i, j] and not protected_periodic_mask[i, j]:
                    final_imp[i, j] = val[i]

    remaining_nans_before_fallback = int(np.isnan(final_imp).sum())
    if remaining_nans_before_fallback > 0:
        col_means = np.nanmean(final_imp, axis=0)
        for j in range(n_cols):
            mask_j = np.isnan(final_imp[:, j])
            if mask_j.any():
                final_imp[mask_j, j] = col_means[j]

    final_imp = np.clip(final_imp, 0, None)
    max_diff = float(np.max(np.abs(original_data[~original_nan_mask] - final_imp[~original_nan_mask])))

    result_df = input_df.copy()
    for j, col in enumerate(cols):
        result_df[col] = final_imp[:, j]

    mechanism_df = pd.DataFrame([
        {'step': 'step_minus1_periodic_prefill', 'count': periodic_prefill_count, 'pct_of_original_nans': 100.0 * periodic_prefill_count / original_nan_total if original_nan_total > 0 else np.nan, 'note': 'Periodic columns filled first and then protected from later edits.' if protected_periodic_cols else 'Periodic prefill not used.'},
        {'step': 'step0_pre_reverse_exact', 'count': pre_filled_total, 'pct_of_original_nans': 100.0 * pre_filled_total / original_nan_total if original_nan_total > 0 else np.nan, 'note': 'Exact algebraic fills before KNN, after the periodic-prefill stage.'},
        {'step': 'step4_long_gap_override', 'count': n_overridden, 'pct_of_original_nans': 100.0 * n_overridden / original_nan_total if original_nan_total > 0 else np.nan, 'note': 'Long-gap cells overwritten with market level plus column mean deviation.'},
        {'step': 'step6_index_reconstruction', 'count': int(sum(original_nan_mask[:, cols.index(idx_col)].sum() for idx_col in index_coefs if idx_col in cols)), 'pct_of_original_nans': 100.0 * sum(original_nan_mask[:, cols.index(idx_col)].sum() for idx_col in index_coefs if idx_col in cols) / original_nan_total if original_nan_total > 0 else np.nan, 'note': 'Original missing index cells eligible for exact reconstruction from known coefficients.'},
        {'step': 'step6_post_knn_reverse', 'count': adj, 'pct_of_original_nans': 100.0 * adj / original_nan_total if original_nan_total > 0 else np.nan, 'note': 'Post-KNN reverse inference with one missing constituent.'},
    ])

    coverage_df = pd.DataFrame([
        {'stage': 'original', 'remaining_nans': original_nan_total, 'pct_of_original_nans_remaining': 100.0},
        {'stage': 'after_periodic_prefill', 'remaining_nans': int(nan_mask.sum()), 'pct_of_original_nans_remaining': 100.0 * nan_mask.sum() / original_nan_total if original_nan_total > 0 else np.nan},
        {'stage': 'after_step0_pre_reverse', 'remaining_nans': int(pre_nan.sum()), 'pct_of_original_nans_remaining': 100.0 * pre_nan.sum() / original_nan_total if original_nan_total > 0 else np.nan},
        {'stage': 'after_step4_long_gap_override', 'remaining_nans': int(np.isnan(final_imp).sum()) if remaining_nans_before_fallback == 0 else remaining_nans_before_fallback, 'pct_of_original_nans_remaining': 100.0 * (int(np.isnan(final_imp).sum()) if remaining_nans_before_fallback == 0 else remaining_nans_before_fallback) / original_nan_total if original_nan_total > 0 else np.nan},
        {'stage': 'after_fallback', 'remaining_nans': int(np.isnan(final_imp).sum()), 'pct_of_original_nans_remaining': 100.0 * np.isnan(final_imp).sum() / original_nan_total if original_nan_total > 0 else np.nan},
    ])

    diagnostics_df = pd.DataFrame([{
        'n_rows': n_rows,
        'n_cols': n_cols,
        'original_nan_fraction': float(original_nan_mask.mean()),
        'original_nan_total': original_nan_total,
        'post_periodic_nan_total': int(nan_mask.sum()),
        'max_diff_on_observed_values': max_diff,
        'imputed_mean': float(np.mean(final_imp[original_nan_mask])) if original_nan_total > 0 else np.nan,
        'imputed_std': float(np.std(final_imp[original_nan_mask])) if original_nan_total > 0 else np.nan,
    }])

    if return_periodic_details:
        return result_df, diagnostics_df, mechanism_df, coverage_df, periodic_prefill_summary_df, periodic_prefill_candidates_df, periodic_prefill_fit_frames
    return result_df, diagnostics_df, mechanism_df, coverage_df




In [5]:
problem2_imputed_df, p2_diag, p2_mech, p2_cov = solve_problem_2(df)
display(p2_diag)
display(p2_mech)
print(f"Remaining NaNs: {problem2_imputed_df[cols].isna().sum().sum()}")

,n_rows,n_cols,original_nan_fraction,original_nan_total,post_periodic_nan_total,max_diff_on_observed_values,imputed_mean,imputed_std
0,3650,53,0.488343,94470,84357,0.0,167.084662,20.735573


,step,count,pct_of_original_nans,note
0,step_minus1_periodic_prefill,10113,10.704986,Periodic columns filled first and then protect...
1,step0_pre_reverse_exact,81925,86.720652,"Exact algebraic fills before KNN, after the pe..."
2,step4_long_gap_override,0,0.000000,Long-gap cells overwritten with market level p...
3,step6_index_reconstruction,82255,87.069969,Original missing index cells eligible for exac...
4,step6_post_knn_reverse,222,0.234995,Post-KNN reverse inference with one missing co...


Remaining NaNs: 0


In [6]:
def submit_answer_2(filled_df: pd.DataFrame) -> None:
    masked_df = pd.read_csv("limestone_data_challenge_2026.data.csv")
    assert sorted(masked_df.columns) == sorted(filled_df.columns)
    assert np.all(masked_df["time"] == filled_df["time"])
    filled_df.fillna(0).to_csv("modelling_prices.submission.2.csv", index=False)

submit_answer_2(problem2_imputed_df)
print("Saved modelling_prices.submission.2.csv")

Saved modelling_prices.submission.2.csv


## Problems 3, 4, 5: Trading Functions (Streaming)

Each function accumulates rows via `_record_row()` into a function-attribute buffer.
Temporal lookback scans this buffer for recent observed values. No training CSV needed at inference.

In [7]:
# All constants for P3/P4/P5 — hardcoded from alt P2 imputed data training.

def _get_constants():
    COLS = [f'col_{i:02d}' for i in range(53)]
    COL_TO_IDX = {c: i for i, c in enumerate(COLS)}
    DEV_MODELS = {
    0: [(18,0.368055,0.03704,0.10408), (48,0.360955,-0.116341,0.099729), (31,-0.077783,0.017928,0.068071), (43,0.280569,-0.186343,0.065625), (16,-0.302747,-0.204301,0.065283), (12,0.096942,-0.089153,0.06196), (24,-0.048917,-0.08245,0.048895), (50,0.233479,-0.073348,0.046717), (1,0.196166,-0.217809,0.041364), (40,-0.24874,-0.09177,0.041206), (28,0.199082,-0.05999,0.040483), (23,-0.189388,0.091109,0.039917), (36,0.185166,-0.045193,0.039165), (27,0.219452,-0.10921,0.037068), (3,0.198054,-0.119664,0.035926), (9,-0.196333,-0.049327,0.035644), (44,-0.223994,-0.168796,0.034542), (25,-0.187875,0.070299,0.0322), (17,-0.217598,-0.176439,0.031266), (32,0.070323,-0.081524,0.030304)],
    1: [(48,0.358005,0.778212,0.091269), (3,0.325055,0.731483,0.090028), (50,0.309697,0.813973,0.076467), (28,0.268266,0.83154,0.068386), (32,0.109524,0.796489,0.068382), (31,-0.074866,0.90931,0.058666), (16,-0.286288,0.697971,0.054309), (43,0.250772,0.721875,0.048772), (24,-0.048527,0.811821,0.044767), (0,0.210861,0.852381,0.041364), (18,0.225789,0.896354,0.03644), (40,-0.241316,0.803429,0.03608), (17,-0.235123,0.707636,0.033961), (23,-0.178973,0.97723,0.033163), (36,0.173335,0.848356,0.031928), (35,-0.15927,0.845711,0.022718), (9,-0.149785,0.843888,0.019301), (27,0.163485,0.799214,0.019139), (49,-0.046898,0.843344,0.016965), (46,-0.122177,0.769952,0.015984)],
    2: [(31,-0.058739,0.110452,0.038388), (32,0.071552,0.02781,0.031024), (7,0.1555,0.141324,0.029674), (0,0.167346,0.065886,0.027694), (49,-0.052437,0.059393,0.022544), (43,0.162937,-0.02051,0.021886), (40,-0.177038,0.029306,0.020642), (24,0.03178,0.076327,0.020409), (50,0.152382,0.043632,0.019678), (52,-0.032387,0.139245,0.019227), (12,0.053859,0.036847,0.018913), (10,-0.117489,0.177221,0.014394), (4,-0.100848,0.218982,0.012762), (48,0.129613,0.034231,0.012716), (18,0.129122,0.088566,0.012668), (51,-0.105967,0.053705,0.011691), (20,-0.124485,0.022257,0.01128), (23,-0.098168,0.131639,0.010606), (16,-0.118337,-0.002171,0.009864), (1,0.090636,-0.019194,0.008732)],
    3: [(32,0.131283,0.284035,0.115312), (50,0.336255,0.308072,0.105797), (1,0.276964,0.104692,0.090028), (28,0.276892,0.327666,0.085504), (31,-0.07599,0.40677,0.070935), (43,0.261111,0.213392,0.062058), (36,0.222422,0.346807,0.061702), (40,-0.284271,0.293134,0.058761), (24,-0.050528,0.307046,0.056961), (16,-0.269433,0.202844,0.056455), (48,0.25981,0.291941,0.056414), (18,0.253134,0.399466,0.053753), (17,-0.258095,0.19102,0.048027), (0,0.181395,0.347263,0.035926), (23,-0.16516,0.463173,0.033145), (27,0.196655,0.287123,0.032501), (44,-0.198298,0.234985,0.029558), (12,0.061561,0.314593,0.027281), (46,-0.136412,0.258083,0.023386), (14,0.149106,0.32627,0.02061)],
    4: [(51,0.46655,1.620405,0.180597), (41,0.293944,1.427722,0.093632), (32,-0.138484,1.662259,0.092611), (26,0.302637,1.560368,0.080447), (37,0.272615,1.709935,0.068636), (8,0.248735,1.685844,0.063166), (31,0.081377,1.531683,0.058717), (35,-0.267386,1.613149,0.054239), (39,0.218906,1.587994,0.053042), (29,-0.245614,1.311962,0.052022), (17,-0.303728,1.433061,0.048007), (46,-0.228928,1.472067,0.04754), (21,-0.229655,1.267989,0.039565), (52,0.050857,1.476595,0.037783), (15,-0.274709,1.355453,0.036362), (23,-0.191456,1.751126,0.032148), (16,-0.225225,1.492942,0.028473), (19,-0.188942,1.540611,0.027721), (20,-0.218044,1.544715,0.027579), (24,-0.038834,1.582111,0.024285)],
    5: [(49,0.264056,0.289779,0.411977), (47,0.518553,0.579776,0.16081), (34,-0.103756,0.047752,0.120244), (31,-0.109798,0.401379,0.09666), (15,0.44237,0.704479,0.085266), (22,-0.259643,0.58297,0.066849), (24,-0.062697,0.26354,0.057242), (46,-0.257647,0.151209,0.054451), (13,0.306962,0.296686,0.051893), (23,-0.250241,0.491691,0.049663), (38,-0.230009,0.248192,0.049231), (30,-0.261938,0.172848,0.045892), (42,-0.311785,0.082414,0.033199), (35,0.219519,0.295416,0.033058), (48,0.217749,0.263222,0.025864), (10,0.142831,0.155476,0.01533), (19,0.139346,0.349539,0.013634), (52,0.031484,0.221663,0.013094), (11,-0.157207,0.277235,0.01303), (51,-0.123692,0.297654,0.011479)],
    6: [(12,0.109427,0.011193,0.081171), (52,-0.062748,0.211489,0.075039), (24,-0.055003,0.018889,0.063561), (31,0.065573,-0.007366,0.049741), (49,-0.068387,0.055298,0.039868), (32,0.079383,0.019803,0.039702), (17,-0.22571,-0.076019,0.034588), (16,-0.211889,-0.053799,0.032879), (50,0.189678,0.03554,0.031701), (46,-0.158684,-0.040356,0.0298), (9,-0.176557,0.055355,0.029637), (18,0.192657,0.099264,0.029321), (22,-0.141123,0.205198,0.028493), (34,-0.037569,-0.039657,0.022746), (26,0.139036,0.031438,0.022152), (51,0.140118,0.056674,0.021252), (30,-0.140503,-0.016796,0.01905), (0,0.136079,0.059429,0.019039), (44,-0.163981,-0.032684,0.019034), (36,0.126686,0.05744,0.01885)],
    7: [(12,0.142233,-0.595292,0.10748), (24,0.07663,-0.495459,0.096693), (52,-0.065014,-0.376939,0.063136), (31,-0.074313,-0.474375,0.050069), (20,-0.244371,-0.610238,0.035421), (2,0.190827,-0.552819,0.029674), (49,-0.063144,-0.539114,0.026639), (19,-0.171272,-0.600899,0.023291), (21,-0.170411,-0.792495,0.022275), (44,-0.175066,-0.632603,0.017003), (3,0.146251,-0.591319,0.015786), (25,-0.142938,-0.448288,0.01502), (18,0.152781,-0.504646,0.014452), (40,-0.149664,-0.565389,0.012021), (43,0.127902,-0.602817,0.01099), (36,0.101467,-0.537772,0.009477), (23,-0.101036,-0.465167,0.009155), (4,-0.093275,-0.392163,0.008896), (35,-0.103473,-0.539034,0.008305), (48,0.1112,-0.561512,0.007627)],
    8: [(26,0.383577,-0.379765,0.126578), (41,0.31994,-0.516035,0.108649), (51,0.340538,-0.311597,0.09424), (32,-0.126148,-0.270802,0.075269), (4,0.25395,-0.730116,0.063166), (35,-0.289799,-0.314243,0.062405), (37,0.249986,-0.22674,0.05653), (12,0.104365,-0.361512,0.055431), (39,0.221051,-0.340198,0.052976), (17,-0.318564,-0.50339,0.051727), (29,-0.239494,-0.608739,0.048446), (46,-0.22996,-0.456555,0.046984), (31,0.06872,-0.38483,0.041012), (21,-0.224501,-0.652452,0.037032), (9,-0.219553,-0.318492,0.034406), (16,-0.249443,-0.447198,0.034209), (20,-0.24524,-0.390907,0.034171), (19,-0.211591,-0.395207,0.034051), (23,-0.16711,-0.195392,0.023989), (45,0.157943,-0.334357,0.023696)],
    9: [(12,-0.118052,0.061895,0.099364), (25,0.294412,-0.175273,0.085512), (44,0.311691,0.179039,0.072331), (30,0.264596,0.147627,0.071062), (34,-0.063669,-0.138143,0.06871), (22,0.211795,-0.211942,0.0675), (29,0.228339,0.290649,0.061698), (24,0.05033,0.048129,0.055977), (16,0.268947,0.152207,0.055715), (49,0.070074,0.014479,0.044028), (13,-0.22924,0.021248,0.043918), (17,0.237676,0.152671,0.04034), (41,-0.156718,0.112476,0.036523), (0,-0.18155,0.008023,0.035644), (8,-0.156711,-0.032911,0.034406), (26,-0.167897,0.042734,0.033977), (39,-0.146371,0.029419,0.032542), (6,-0.167861,0.026376,0.029637), (48,-0.182915,0.049814,0.027696), (51,-0.145942,0.012994,0.024249)],
    10: [(49,0.11964,1.017457,0.112545), (13,0.388815,1.016619,0.110794), (30,-0.320027,0.865537,0.09116), (42,-0.441723,0.712315,0.088677), (22,-0.256149,1.300416,0.08658), (33,0.33288,0.943388,0.084518), (52,-0.068087,1.195589,0.081492), (17,0.341837,1.217054,0.073175), (21,0.272079,1.422846,0.066824), (34,0.065972,1.18418,0.064693), (15,0.306419,1.301886,0.054441), (20,0.278734,1.100708,0.054233), (38,-0.205998,0.974997,0.052549), (35,0.227071,1.016438,0.04707), (19,0.214425,1.096621,0.042963), (11,-0.245364,0.984826,0.04224), (47,0.224412,1.143197,0.040078), (48,-0.21096,1.059943,0.032305), (46,0.171886,1.123103,0.03225), (23,0.170294,0.893411,0.030606)],
    11: [(12,-0.106678,-0.114732,0.101411), (30,0.245183,-0.034271,0.076262), (42,0.334861,0.080616,0.072633), (13,-0.216669,-0.15131,0.049036), (22,0.161105,-0.329362,0.048814), (10,-0.172154,0.021325,0.04224), (49,-0.054001,-0.152343,0.03268), (15,-0.197631,-0.334758,0.032277), (24,0.031755,-0.135495,0.02785), (47,-0.150281,-0.235382,0.025617), (38,0.117949,-0.127384,0.024554), (33,-0.147479,-0.119572,0.023644), (41,-0.10173,-0.09317,0.019234), (50,0.128669,-0.166086,0.019177), (28,0.11624,-0.158961,0.018654), (25,0.116108,-0.230819,0.016622), (8,-0.091953,-0.184395,0.014806), (5,-0.082887,-0.129757,0.01303), (32,0.039009,-0.170696,0.012603), (52,0.022349,-0.211472,0.012514)],
    12: [(52,-0.283929,1.095724,0.226652), (30,-1.079552,-0.155324,0.165911), (22,-0.76271,1.201803,0.122774), (25,-0.931028,0.98511,0.119938), (7,0.755665,0.784682,0.10748), (11,-0.950634,0.228049,0.101411), (9,-0.841699,0.389982,0.099364), (38,-0.679432,0.217507,0.09143), (20,-0.879437,0.129345,0.086347), (44,-0.895159,-0.088463,0.083675), (6,0.74178,0.336407,0.081171), (51,0.677773,0.396581,0.073355), (0,0.639147,0.408899,0.06196), (26,0.603491,0.284842,0.061567), (37,0.58388,0.598492,0.060596), (46,-0.564444,0.045772,0.055622), (8,0.531128,0.546376,0.055431), (41,0.507514,0.067937,0.05372), (13,0.665515,0.364588,0.051916), (21,-0.547641,-0.430059,0.0433)],
    13: [(30,-0.326811,-0.144704,0.129717), (22,-0.25535,0.292643,0.117403), (31,-0.088088,0.095969,0.112969), (10,0.284952,-0.275558,0.110794), (34,0.067938,0.182081,0.093611), (38,-0.234107,-0.038433,0.092606), (42,-0.379403,-0.250789,0.089265), (15,0.331289,0.317631,0.086832), (49,0.088896,0.011922,0.084782), (21,0.233981,0.359922,0.067434), (20,0.263589,0.089567,0.066177), (47,0.221068,0.134496,0.053069), (12,0.078008,-0.013377,0.051916), (5,0.169052,-0.035091,0.051893), (25,-0.209328,0.153027,0.051725), (46,0.183881,0.123196,0.050361), (11,-0.226318,-0.019134,0.049036), (9,-0.191583,0.019262,0.043918), (51,-0.159065,0.010862,0.034469), (24,-0.03372,-0.004561,0.030065)],
    14: [(24,-0.04261,0.050739,0.043696), (27,0.179284,0.030482,0.029139), (46,-0.146271,-0.008779,0.029005), (17,-0.187003,-0.029689,0.027198), (16,-0.176709,-0.011858,0.026195), (3,0.138225,0.029903,0.02061), (18,0.147916,0.112679,0.019799), (35,-0.125791,0.080103,0.01794), (6,0.12307,0.07015,0.01735), (34,-0.030478,0.002023,0.017149), (23,-0.109344,0.159657,0.015672), (22,-0.092526,0.176862,0.014031), (8,0.095666,0.107419,0.013965), (36,0.101753,0.080751,0.01393), (26,0.09869,0.06181,0.012785), (48,0.117723,0.055851,0.012494), (12,0.040035,0.06156,0.012446), (1,0.097176,-0.005169,0.011955), (28,0.097231,0.073061,0.011373), (40,-0.119639,0.057829,0.011228)],
    15: [(49,0.099827,-0.915267,0.135138), (22,-0.215823,-0.676899,0.106006), (47,0.273787,-0.76392,0.102885), (30,-0.245078,-1.031241,0.092203), (13,0.262104,-0.914976,0.086832), (5,0.192747,-0.968937,0.085266), (24,-0.048057,-0.939956,0.077184), (21,0.195797,-0.622922,0.059685), (10,0.177669,-1.09253,0.054441), (52,-0.041391,-0.805769,0.051939), (20,0.200185,-0.854856,0.048245), (42,-0.244461,-1.08264,0.046842), (38,-0.144905,-0.944435,0.044845), (26,-0.153455,-0.887845,0.042926), (31,-0.048213,-0.866982,0.042774), (19,0.155665,-0.857217,0.039051), (35,0.152748,-0.915089,0.036735), (4,-0.132366,-0.698276,0.036362), (51,-0.142733,-0.915322,0.03508), (11,-0.163322,-0.936086,0.032277)],
    16: [(17,0.353036,-0.299853,0.115549), (49,0.090868,-0.50453,0.096116), (48,-0.268909,-0.453125,0.077712), (0,-0.215635,-0.511857,0.065283), (28,-0.212132,-0.492796,0.064533), (18,-0.239712,-0.558976,0.061984), (35,0.209239,-0.506334,0.059171), (3,-0.209532,-0.429718,0.056455), (9,0.207158,-0.504122,0.055715), (1,-0.189701,-0.340889,0.054309), (44,0.235946,-0.378272,0.053811), (43,-0.198542,-0.405963,0.046137), (29,0.166861,-0.300946,0.042774), (24,0.038468,-0.477145,0.042455), (41,-0.143139,-0.413825,0.039556), (12,-0.064908,-0.476124,0.038998), (36,-0.155119,-0.506834,0.038589), (51,-0.16151,-0.505578,0.038557), (27,-0.185248,-0.452842,0.037084), (8,-0.137141,-0.544683,0.034209)],
    17: [(16,0.3273,-0.40447,0.115549), (51,-0.231812,-0.575601,0.085675), (48,-0.261273,-0.52227,0.07913), (21,0.225624,-0.23653,0.073382), (10,0.214064,-0.787219,0.073175), (23,0.20458,-0.723711,0.070537), (41,-0.183515,-0.457183,0.07013), (49,0.073709,-0.571565,0.068217), (52,-0.049031,-0.443844,0.067483), (39,-0.175254,-0.554131,0.065329), (26,-0.195283,-0.539048,0.064366), (35,0.204605,-0.574004,0.061028), (46,0.177777,-0.464531,0.05509), (8,-0.162375,-0.620618,0.051727), (3,-0.186083,-0.505437,0.048027), (4,-0.158059,-0.314486,0.048007), (18,-0.203018,-0.617822,0.047956), (29,0.163018,-0.373342,0.044037), (9,0.169726,-0.571263,0.04034), (28,-0.15926,-0.56251,0.039233)],
    18: [(48,0.334507,-0.302951,0.111478), (0,0.282784,-0.229123,0.10408), (43,0.24257,-0.359521,0.063844), (16,-0.258579,-0.373462,0.061984), (32,0.087173,-0.279677,0.060607), (50,0.232852,-0.264559,0.060477), (24,-0.04585,-0.271856,0.05591), (31,-0.0609,-0.188687,0.054311), (3,0.21235,-0.315758,0.053753), (28,0.195223,-0.251117,0.050667), (36,0.18037,-0.236656,0.048369), (17,-0.236216,-0.378286,0.047956), (27,0.210075,-0.298066,0.044211), (23,-0.1741,-0.111772,0.043905), (12,0.070875,-0.27064,0.043106), (40,-0.217682,-0.278168,0.041074), (46,-0.163867,-0.339677,0.040228), (1,0.161388,-0.379818,0.03644), (25,-0.160565,-0.138858,0.030611), (6,0.152191,-0.252001,0.029321)],
    19: [(35,0.304955,-0.352831,0.090856), (49,0.096388,-0.348593,0.078177), (20,0.296367,-0.261452,0.065615), (51,-0.206122,-0.350805,0.045395), (10,0.200362,-0.549222,0.042963), (26,-0.194789,-0.315139,0.042919), (52,-0.047637,-0.223398,0.04269), (21,0.206245,-0.04104,0.041093), (15,0.250863,-0.115803,0.039051), (8,-0.160929,-0.396169,0.034051), (39,-0.148525,-0.332304,0.031445), (23,0.166695,-0.470943,0.031385), (41,-0.149163,-0.253995,0.031051), (12,-0.064988,-0.319911,0.02826), (4,-0.146716,-0.108715,0.027721), (40,0.199153,-0.313078,0.026808), (48,-0.178538,-0.312854,0.024763), (7,-0.135991,-0.41799,0.023291), (42,-0.217997,-0.497519,0.023114), (17,0.179101,-0.242513,0.021497)],
    20: [(21,0.330879,0.206989,0.14158), (46,0.239188,-0.139935,0.089464), (12,-0.098184,-0.242682,0.086347), (24,-0.050965,-0.310425,0.072106), (51,-0.217577,-0.286393,0.067709), (34,0.055766,-0.143101,0.06622), (13,0.251062,-0.283506,0.066177), (19,0.221396,-0.203292,0.065615), (35,0.211532,-0.285441,0.058519), (26,-0.191955,-0.250789,0.055792), (10,0.194568,-0.478521,0.054233), (23,0.184995,-0.420073,0.051743), (41,-0.163306,-0.180659,0.049821), (15,0.241001,-0.060011,0.048245), (49,0.062629,-0.282312,0.044182), (7,-0.144949,-0.35807,0.035421), (8,-0.139339,-0.324434,0.034171), (17,0.190166,-0.171451,0.032442), (4,-0.126484,-0.076427,0.027579), (22,-0.118808,-0.150751,0.026683)],
    21: [(20,0.42789,-1.350744,0.14158), (46,0.297195,-1.296914,0.106804), (34,0.076883,-1.282273,0.09733), (23,0.258782,-1.666963,0.078295), (51,-0.264272,-1.478698,0.077243), (17,0.32524,-1.285521,0.073382), (13,0.288202,-1.474926,0.067434), (10,0.245606,-1.721552,0.066824), (41,-0.210025,-1.343207,0.063721), (24,-0.053107,-1.502554,0.060544), (15,0.304829,-1.192705,0.059685), (35,0.230787,-1.476809,0.053864), (26,-0.200314,-1.440367,0.046982), (12,-0.079067,-1.440684,0.0433), (19,0.199245,-1.401748,0.041093), (52,-0.04577,-1.354191,0.040794), (39,-0.163914,-1.457118,0.039644), (4,-0.172278,-1.193726,0.039565), (22,-0.162476,-1.294252,0.038589), (33,0.209692,-1.520369,0.037153)],
    22: [(24,0.122082,1.157857,0.218868), (30,0.513026,1.335918,0.177533), (38,0.367271,1.169042,0.126585), (12,-0.160971,1.14421,0.122774), (13,-0.459771,1.091126,0.117403), (15,-0.491173,0.636454,0.106006), (47,-0.392318,0.873335,0.092825), (10,-0.338007,1.429533,0.08658), (33,-0.361288,1.170005,0.075448), (49,-0.110741,1.088762,0.073073), (9,0.318704,1.078209,0.0675), (5,-0.257465,1.161462,0.066849), (42,0.429089,1.385422,0.063412), (11,0.302995,1.13071,0.048814), (21,-0.237508,0.734601,0.038589), (25,0.233726,0.930699,0.035814), (39,-0.17339,1.097814,0.030347), (6,-0.201899,1.094369,0.028493), (52,0.045841,0.967484,0.027994), (20,-0.224593,1.021043,0.026683)],
    23: [(34,0.117422,1.047021,0.194187), (46,0.426005,1.00838,0.187701), (21,0.302553,1.204635,0.078295), (17,0.344787,0.955712,0.070537), (39,-0.231072,0.778427,0.067387), (52,-0.062791,0.919129,0.065669), (26,-0.237335,0.795298,0.056411), (48,-0.282215,0.809471,0.05478), (20,0.279699,0.837959,0.051743), (41,-0.204412,0.88352,0.051629), (5,-0.198462,0.819627,0.049663), (18,-0.252181,0.698233,0.043905), (51,-0.214876,0.752988,0.043679), (0,-0.210769,0.748653,0.039917), (28,-0.199792,0.76701,0.036636), (12,-0.077307,0.788781,0.035405), (1,-0.185297,0.915659,0.033163), (3,-0.200687,0.827548,0.033145), (4,-0.167915,1.029393,0.032148), (19,0.188276,0.8246,0.031385)],
    24: [(22,1.792798,-2.549526,0.218868), (30,1.52977,0.145265,0.107491), (7,1.261814,0.077362,0.096693), (50,-1.389537,-0.484066,0.080977), (32,-0.515659,-0.39571,0.079739), (15,-1.606096,-2.069306,0.077184), (20,-1.414822,-1.001922,0.072106), (47,-1.322778,-1.31615,0.071859), (6,-1.155588,-0.54608,0.063561), (28,-1.120162,-0.565907,0.062721), (21,-1.140039,-2.282708,0.060544), (5,-0.912994,-0.33113,0.057242), (3,-1.127319,-0.225773,0.056961), (9,1.112193,-0.626037,0.055977), (18,-1.219408,-0.904052,0.05591), (48,-1.177532,-0.399113,0.051941), (27,-1.142095,-0.312793,0.049133), (0,-0.999566,-0.659216,0.048895), (1,-0.922505,0.169602,0.044767), (14,-1.025496,-0.527923,0.043696)],
    25: [(12,-0.128823,0.703463,0.119938), (34,-0.071871,0.479321,0.088748), (9,0.29045,0.65002,0.085512), (44,0.29382,0.807311,0.065152), (13,-0.247102,0.65906,0.051725), (49,0.071017,0.651964,0.045837), (31,0.055083,0.605059,0.037421), (22,0.153232,0.489057,0.035814), (30,0.184417,0.745755,0.034991), (29,0.168643,0.856794,0.034114), (0,-0.171391,0.646087,0.0322), (18,-0.190646,0.608607,0.030611), (36,-0.142183,0.649305,0.025314), (16,0.168986,0.739707,0.022296), (28,-0.139452,0.660182,0.021774), (46,-0.131129,0.578611,0.021696), (43,-0.153743,0.72832,0.0216), (40,0.162896,0.680665,0.019372), (48,-0.150947,0.681713,0.019118), (3,-0.136151,0.70111,0.018611)],
    26: [(41,0.359048,-0.067688,0.159052), (51,0.37757,0.161595,0.134662), (8,0.329994,0.25604,0.126578), (32,-0.15078,0.211286,0.124994), (39,0.280718,0.127007,0.099308), (35,-0.32809,0.158851,0.092972), (4,0.26582,-0.277153,0.080447), (46,-0.278544,-0.012885,0.080127), (29,-0.283528,-0.189373,0.078924), (31,0.083451,0.0738,0.0703), (17,-0.329606,-0.037643,0.064366), (12,0.102019,0.11139,0.061567), (37,0.240214,0.241543,0.060672), (23,-0.237687,0.330253,0.056411), (20,-0.290651,0.068421,0.055792), (21,-0.234543,-0.195196,0.046982), (15,-0.279731,-0.105119,0.042926), (19,-0.220333,0.073804,0.042919), (16,-0.230901,0.034103,0.034072), (9,-0.202365,0.153226,0.033977)],
    27: [(24,-0.04302,0.231036,0.049133), (18,0.210454,0.308487,0.044211), (46,-0.157952,0.164951,0.037309), (16,-0.200188,0.156937,0.037084), (0,0.168913,0.266042,0.037068), (3,0.165268,0.201317,0.032501), (14,0.16253,0.24468,0.029139), (17,-0.175066,0.157641,0.026293), (44,-0.167242,0.170507,0.025018), (12,0.052938,0.237266,0.024005), (8,0.118373,0.295285,0.023584), (49,-0.045576,0.25916,0.022375), (9,-0.133183,0.259471,0.02131), (32,0.050764,0.236379,0.020516), (35,-0.123847,0.260594,0.019183), (1,0.117066,0.158645,0.019139), (36,0.113151,0.261765,0.019001), (43,0.130924,0.194802,0.018565), (6,0.120283,0.250842,0.018282), (40,-0.144498,0.234479,0.018066)],
    28: [(50,0.378281,0.00288,0.120059), (48,0.375233,-0.029873,0.105514), (32,0.132094,-0.017787,0.104679), (3,0.3088,-0.06808,0.085504), (1,0.254918,-0.178252,0.068386), (16,-0.304211,-0.116051,0.064533), (24,-0.055993,0.002241,0.062721), (31,-0.072511,0.102117,0.057915), (18,0.259536,0.099538,0.050667), (36,0.212831,0.044923,0.050657), (43,0.247271,-0.08151,0.049903), (40,-0.26105,-0.004717,0.044433), (0,0.203348,0.046932,0.040483), (17,-0.246347,-0.103795,0.039233), (23,-0.183372,0.17552,0.036636), (25,-0.15614,0.138491,0.021774), (11,0.160478,0.061033,0.018654), (10,-0.133905,0.173156,0.018511), (44,-0.16394,-0.04871,0.018115), (9,-0.140106,0.038665,0.017771)],
    29: [(52,-0.076348,-1.002021,0.098743), (26,-0.278363,-1.154116,0.078924), (34,-0.073774,-1.376247,0.07796), (39,-0.23448,-1.176853,0.070573), (41,-0.234263,-1.053965,0.068965), (9,0.270202,-1.200534,0.061698), (32,0.104214,-1.238368,0.060819), (51,-0.250639,-1.203698,0.060441), (4,-0.211803,-0.855693,0.052022), (49,0.08032,-1.199362,0.048882), (35,0.234971,-1.202357,0.048572), (8,-0.202285,-1.260985,0.048446), (17,0.270134,-1.042267,0.044037), (16,0.256347,-1.067482,0.042774), (44,0.242305,-1.070281,0.03694), (25,0.202284,-1.3283,0.034114), (24,0.03885,-1.172217,0.028185), (37,-0.149101,-1.252807,0.023809), (47,0.162247,-1.108729,0.020188), (19,0.131679,-1.150441,0.015614)],
    30: [(22,0.34605,-0.866451,0.177533), (12,-0.153685,-0.433738,0.165911), (13,-0.396918,-0.485088,0.129717), (24,0.070266,-0.448781,0.107491), (15,-0.376219,-0.834059,0.092203), (10,-0.284851,-0.20005,0.09116), (47,-0.306036,-0.655589,0.08374), (38,0.243834,-0.434815,0.082717), (11,0.311041,-0.44326,0.076262), (33,-0.295328,-0.420945,0.074739), (42,0.376619,-0.226673,0.072424), (9,0.268567,-0.496123,0.071062), (49,-0.084003,-0.487646,0.062335), (52,0.049358,-0.616656,0.048114), (5,-0.175202,-0.43856,0.045892), (25,0.189739,-0.615699,0.034991), (0,-0.155838,-0.49962,0.025875), (37,-0.127646,-0.540218,0.020344), (6,-0.135588,-0.484311,0.01905), (43,-0.135467,-0.426908,0.0163)],
    31: [(51,1.123131,0.944577,0.118036), (13,-1.282454,0.929462,0.112969), (5,-0.880346,1.174565,0.09666), (48,-1.050127,1.093993,0.075025), (3,-0.933487,1.224313,0.070935), (26,0.842415,0.783006,0.0703), (0,-0.875143,0.862892,0.068071), (4,0.721547,-0.249475,0.058717), (1,-0.783616,1.568303,0.058666), (28,-0.798711,0.937997,0.057915), (43,-0.872649,1.324493,0.056425), (41,0.66856,0.504369,0.054628), (18,-0.891801,0.691441,0.054311), (7,-0.673758,0.543954,0.050069), (52,-0.174014,1.3507,0.049888), (6,0.758549,0.869454,0.049741), (36,-0.676056,0.88137,0.046404), (15,-0.887194,0.101019,0.042774), (8,0.596805,1.10147,0.041012), (49,-0.2291,0.919309,0.038679)],
    32: [(50,1.534735,0.273513,0.329414), (26,-0.828984,0.53276,0.124994), (41,-0.728249,0.84954,0.119012), (3,0.878349,0.112083,0.115312), (51,-0.806759,0.383197,0.111824), (28,0.792456,0.380006,0.104679), (39,-0.668086,0.462611,0.102306), (4,-0.668753,1.482482,0.092611), (48,0.816717,0.264882,0.083322), (24,-0.154635,0.314912,0.079739), (8,-0.596675,0.216348,0.075269), (52,-0.151725,0.793742,0.069637), (1,0.624359,-0.116552,0.068382), (29,0.583596,1.106542,0.060819), (18,0.695246,0.578366,0.060607), (43,0.657424,0.095737,0.0588), (37,-0.534126,0.204392,0.05456), (36,0.535564,0.430646,0.053469), (49,-0.179338,0.416695,0.043517), (12,-0.194886,0.481805,0.040865)],
    33: [(52,-0.08071,0.443377,0.150132), (10,0.253898,-0.021138,0.084518), (22,-0.20883,0.464883,0.075448), (30,-0.253072,0.11419,0.074739), (12,0.070327,0.212164,0.040544), (47,0.195327,0.343345,0.039808), (24,-0.038408,0.215256,0.037478), (21,0.177178,0.499062,0.037153), (38,-0.149591,0.203837,0.036331), (13,0.173427,0.235793,0.028899), (31,0.044534,0.198064,0.027743), (42,-0.211328,0.090009,0.02661), (49,0.049807,0.236326,0.025573), (11,-0.160324,0.213738,0.023644), (35,0.135428,0.234756,0.021952), (19,0.127068,0.282297,0.019781), (15,0.15815,0.382594,0.019014), (17,0.147374,0.322297,0.017832), (23,0.092369,0.168369,0.011806), (6,0.094745,0.233598,0.010855)],
    34: [(46,2.590858,-0.934302,0.492952), (23,1.653749,-3.702722,0.194187), (47,-1.551686,-3.278747,0.128912), (49,-0.481436,-2.424754,0.122608), (5,-1.158915,-2.096753,0.120244), (21,1.265948,-0.584857,0.09733), (13,1.377895,-2.468133,0.093611), (25,-1.234831,-1.63726,0.088748), (29,-1.05674,-3.709866,0.07796), (9,-1.079186,-2.42724,0.06871), (20,1.187459,-2.114325,0.06622), (10,0.980603,-3.449197,0.064693), (44,-1.156589,-3.045266,0.058758), (42,-1.159068,-3.260935,0.041077), (48,-0.79447,-2.306348,0.030825), (50,-0.724298,-2.382444,0.028684), (17,0.820038,-1.980233,0.028331), (18,-0.724073,-2.62295,0.0257), (51,-0.617528,-2.465754,0.025615), (6,-0.605437,-2.414608,0.022746)],
    35: [(49,0.143971,0.021577,0.178525), (51,-0.30373,0.018404,0.100892), (26,-0.283374,0.070413,0.092972), (19,0.297933,0.130578,0.090856), (41,-0.238471,0.172362,0.081234), (8,-0.215337,-0.041413,0.062405), (17,0.298274,0.197504,0.061028), (52,-0.055775,0.169549,0.059902), (16,0.282793,0.169533,0.059171), (20,0.276642,0.105329,0.058519), (37,-0.211155,-0.052763,0.054278), (4,-0.202848,0.353708,0.054239), (21,0.233394,0.371172,0.053864), (39,-0.189252,0.043277,0.052258), (29,0.206713,0.275186,0.048572), (10,0.207293,-0.184016,0.04707), (48,-0.222139,0.067117,0.039238), (12,-0.075621,0.056373,0.039165), (15,0.240494,0.247047,0.036735), (5,0.150593,-0.017411,0.033058)],
    36: [(50,0.321904,-0.069347,0.077741), (3,0.277408,-0.134672,0.061702), (32,0.099837,-0.081797,0.053469), (28,0.238017,-0.04961,0.050657), (18,0.268165,0.024451,0.048369), (31,-0.068639,0.021404,0.046404), (0,0.211515,-0.02983,0.039165), (16,-0.248774,-0.165499,0.038589), (48,0.232985,-0.082019,0.036374), (43,0.212154,-0.141986,0.032848), (24,-0.042374,-0.066693,0.03212), (1,0.184201,-0.195954,0.031928), (12,0.07305,-0.0684,0.0308), (17,-0.23062,-0.17205,0.030745), (40,-0.223899,-0.076087,0.029228), (44,-0.21694,-0.153353,0.028365), (25,-0.178036,0.075643,0.025314), (23,-0.150824,0.073598,0.022162), (27,0.167924,-0.084172,0.019001), (6,0.148789,-0.048768,0.01885)],
    37: [(51,0.292631,-0.373245,0.076931), (4,0.25177,-0.786749,0.068636), (26,0.252573,-0.420293,0.060672), (12,0.103782,-0.421428,0.060596), (8,0.22613,-0.309598,0.05653), (32,-0.102148,-0.340745,0.05456), (35,-0.257053,-0.375294,0.054278), (46,-0.203489,-0.501242,0.040671), (41,0.17779,-0.490119,0.03709), (39,0.173376,-0.396485,0.036027), (23,-0.176303,-0.248542,0.029518), (31,0.055281,-0.432747,0.02934), (21,-0.176837,-0.642505,0.025401), (29,-0.159681,-0.573436,0.023809), (16,-0.195639,-0.480405,0.023263), (30,-0.159375,-0.460808,0.020344), (19,-0.151595,-0.434686,0.019323), (45,0.126856,-0.39213,0.016899), (20,-0.158842,-0.426892,0.015848), (6,0.138165,-0.389711,0.015843)],
    38: [(22,0.344664,-0.605594,0.126585), (13,-0.395574,-0.225755,0.092606), (12,-0.134568,-0.181555,0.09143), (30,0.339237,-0.065341,0.082717), (49,-0.097026,-0.22771,0.059773), (10,-0.255096,0.028872,0.052549), (47,-0.284919,-0.384904,0.05217), (5,-0.21404,-0.167494,0.049231), (15,-0.309478,-0.513916,0.044845), (24,0.049846,-0.201811,0.038881), (33,-0.24287,-0.174104,0.036331), (42,0.311842,-0.01285,0.035689), (32,0.0755,-0.262896,0.026749), (31,0.053885,-0.281026,0.025017), (11,0.208177,-0.199824,0.024554), (9,0.182955,-0.235261,0.023703), (0,-0.150951,-0.240008,0.01745), (25,0.157774,-0.335403,0.01739), (44,0.159562,-0.149399,0.013423), (18,-0.128352,-0.263364,0.009693)],
    39: [(32,-0.153133,0.143292,0.102306), (51,0.36705,0.092307,0.100986), (26,0.353762,0.027763,0.099308), (41,0.278469,-0.087864,0.075918), (29,-0.300976,-0.279192,0.070573), (52,0.071497,-0.100738,0.067463), (23,-0.291629,0.302282,0.067387), (17,-0.372767,-0.131126,0.065329), (4,0.242305,-0.308352,0.053042), (8,0.239655,0.157963,0.052976), (35,-0.27613,0.08844,0.052258), (46,-0.244953,-0.062238,0.049172), (21,-0.241861,-0.274912,0.039644), (37,0.207798,0.160189,0.036027), (12,0.084809,0.048891,0.033762), (16,-0.253656,-0.04624,0.032628), (9,-0.222326,0.084622,0.032542), (19,-0.211717,0.007816,0.031445), (22,-0.17502,0.270398,0.030347), (24,-0.043309,0.054443,0.027288)],
    40: [(48,-0.242147,-0.114096,0.067392), (3,-0.206709,-0.08693,0.058761), (28,-0.17021,-0.150572,0.044433), (0,-0.165659,-0.165477,0.041206), (18,-0.188689,-0.202783,0.041074), (32,-0.064638,-0.130316,0.038442), (50,-0.171553,-0.141623,0.037871), (31,0.046335,-0.198855,0.036269), (1,-0.149513,-0.030955,0.03608), (10,0.138811,-0.298709,0.030509), (23,0.134919,-0.259242,0.030418), (36,-0.130539,-0.162085,0.029228), (43,-0.150434,-0.085122,0.028328), (19,0.134609,-0.110389,0.026808), (49,0.04578,-0.158776,0.026091), (17,0.161199,-0.065128,0.025765), (35,0.11966,-0.160084,0.020696), (2,-0.116596,-0.150081,0.020642), (52,-0.027167,-0.087787,0.020543), (25,0.118921,-0.234643,0.019372)],
    41: [(51,0.476616,0.620416,0.173922), (26,0.442982,0.539056,0.159052), (32,-0.163422,0.672143,0.119012), (8,0.33959,0.714824,0.108649), (4,0.318538,0.09389,0.093632), (35,-0.340643,0.614893,0.081234), (39,0.272627,0.583351,0.075918), (17,-0.382151,0.388188,0.07013), (29,-0.29439,0.25333,0.068965), (21,-0.303399,0.159253,0.063721), (46,-0.262286,0.452293,0.057585), (31,0.08171,0.531073,0.054628), (12,0.10585,0.565644,0.05372), (23,-0.252571,0.797252,0.051629), (20,-0.305077,0.52008,0.049821), (52,0.058512,0.456861,0.046152), (16,-0.276344,0.467051,0.039556), (37,0.208618,0.685149,0.03709), (9,-0.233047,0.609457,0.036523), (50,-0.226857,0.625336,0.032074)],
    42: [(13,-0.235279,-0.699149,0.089265), (10,-0.200752,-0.497559,0.088677), (49,-0.067335,-0.699883,0.078442), (11,0.216905,-0.669321,0.072633), (30,0.1923,-0.608392,0.072424), (22,0.147783,-0.863058,0.063412), (15,-0.191613,-0.877411,0.046842), (34,-0.035439,-0.789581,0.041077), (38,0.114447,-0.676331,0.035689), (24,0.028507,-0.6856,0.03465), (21,-0.131565,-0.896334,0.034381), (5,-0.106481,-0.670777,0.033199), (47,-0.127831,-0.771471,0.028614), (46,-0.105526,-0.764469,0.026746), (33,-0.12592,-0.67285,0.02661), (20,-0.126469,-0.738238,0.024566), (19,-0.106028,-0.739392,0.023114), (32,0.040918,-0.719611,0.021408), (50,0.095788,-0.711325,0.016408), (17,-0.106977,-0.76368,0.015769)],
    43: [(0,0.233898,0.488377,0.065625), (18,0.263199,0.540264,0.063844), (50,0.248188,0.454171,0.063321), (3,0.237669,0.395773,0.062058), (32,0.08944,0.439477,0.0588), (31,-0.06466,0.534812,0.056425), (48,0.242118,0.433398,0.053825), (28,0.201815,0.468725,0.049903), (1,0.194489,0.312417,0.048772), (16,-0.232381,0.35973,0.046137), (12,0.073401,0.448494,0.042609), (36,0.154831,0.482378,0.032848), (40,-0.188307,0.446517,0.028328), (44,-0.17724,0.384234,0.025943), (24,-0.032353,0.45641,0.025657), (46,-0.134978,0.397262,0.025155), (23,-0.136112,0.579446,0.024732), (17,-0.175448,0.376328,0.024382), (9,-0.14474,0.478579,0.023238), (34,-0.034724,0.391087,0.02267)],
    44: [(12,-0.093475,-0.482857,0.083675), (9,0.23206,-0.522011,0.072331), (25,0.22174,-0.663194,0.065152), (34,-0.050803,-0.6422,0.058758), (16,0.228063,-0.403785,0.053811), (49,0.064799,-0.520817,0.050567), (29,0.152451,-0.335627,0.03694), (17,0.190553,-0.409639,0.034827), (0,-0.154211,-0.526065,0.034542), (48,-0.164562,-0.488948,0.030109), (3,-0.149059,-0.467589,0.029558), (36,-0.130749,-0.523285,0.028365), (18,-0.154862,-0.555719,0.026764), (43,-0.146371,-0.448248,0.025943), (27,-0.14959,-0.479461,0.025018), (22,0.106591,-0.633451,0.022964), (31,0.037213,-0.551755,0.022632), (24,0.026476,-0.501868,0.020806), (13,-0.133235,-0.515808,0.019926), (6,-0.116074,-0.51186,0.019034)],
    45: [(51,0.19185,0.082032,0.031488), (26,0.179414,0.049118,0.029153), (8,0.15003,0.124333,0.023696), (32,-0.067572,0.103585,0.022736), (39,0.139481,0.064712,0.022205), (46,-0.151649,-0.012528,0.02151), (41,0.126695,-0.000726,0.017936), (37,0.133214,0.126922,0.016899), (4,0.124774,-0.124376,0.016053), (31,0.04019,0.039433,0.014768), (23,-0.127681,0.172979,0.014743), (35,-0.118621,0.079291,0.011007), (19,-0.116437,0.035881,0.010855), (24,-0.025464,0.060527,0.010767), (17,-0.136172,-0.001414,0.00995), (12,0.042494,0.060027,0.009674), (16,-0.127286,0.012266,0.009377), (29,-0.098855,-0.042239,0.008689), (34,-0.025997,0.012374,0.008608), (21,-0.099938,-0.070974,0.007725)],
    46: [(34,0.190266,-0.11813,0.492952), (23,0.440607,-0.91833,0.187701), (21,0.359373,-0.055163,0.106804), (51,-0.31851,-0.593632,0.09279), (20,0.374031,-0.479018,0.089464), (26,-0.287665,-0.540513,0.080127), (47,-0.329773,-0.760495,0.079287), (41,-0.219551,-0.45066,0.057585), (12,-0.098543,-0.546597,0.055622), (17,0.309884,-0.407467,0.05509), (5,-0.211341,-0.519834,0.054451), (13,0.273878,-0.587918,0.050361), (39,-0.200741,-0.567365,0.049172), (4,-0.207662,-0.250131,0.04754), (8,-0.204315,-0.649429,0.046984), (37,-0.199869,-0.660015,0.040671), (18,-0.245494,-0.643479,0.040228), (48,-0.241432,-0.541054,0.038763), (27,-0.236206,-0.522832,0.037309), (10,0.187624,-0.775468,0.03225)],
    47: [(49,0.167875,-0.54401,0.278435), (5,0.310113,-0.630037,0.16081), (34,-0.083079,-0.739749,0.128912), (15,0.375783,-0.19425,0.102885), (22,-0.236606,-0.28008,0.092825), (30,-0.273627,-0.670977,0.08374), (46,-0.240428,-0.676824,0.079287), (24,-0.054324,-0.569463,0.071859), (13,0.240058,-0.540332,0.053069), (38,-0.183104,-0.579006,0.05217), (10,0.178593,-0.719182,0.040078), (33,0.203803,-0.585135,0.039808), (42,-0.223845,-0.693856,0.028614), (52,-0.035978,-0.445212,0.028592), (23,-0.14472,-0.426563,0.027775), (12,0.057946,-0.558257,0.02638), (11,-0.170459,-0.562897,0.025617), (35,0.141902,-0.540492,0.023098), (29,0.12443,-0.387727,0.020188), (19,0.121504,-0.494685,0.017334)],
    48: [(18,0.333259,0.257414,0.111478), (28,0.281197,0.165903,0.105514), (0,0.276294,0.190666,0.099729), (1,0.254938,-0.038385,0.091269), (50,0.277804,0.151613,0.086404), (32,0.102021,0.134387,0.083322), (17,-0.302863,0.003973,0.07913), (16,-0.288991,0.03145,0.077712), (31,-0.071444,0.241031,0.075025), (40,-0.27831,0.132462,0.067392), (3,0.217137,0.102757,0.056414), (23,-0.194109,0.323561,0.05478), (43,0.222309,0.070256,0.053825), (24,-0.04411,0.149331,0.051941), (35,-0.176635,0.181028,0.039238), (46,-0.160554,0.082388,0.038763), (36,0.156122,0.182482,0.036374), (10,-0.153135,0.332708,0.032305), (34,-0.038799,0.08117,0.030825), (44,-0.182965,0.08132,0.030109)],
    49: [(5,1.560189,-0.425868,0.411977), (47,1.658589,0.934491,0.278435), (35,1.240008,0.009904,0.178525), (15,1.353717,1.277609,0.135138), (34,-0.254671,-0.578359,0.122608), (10,0.940695,-0.917512,0.112545), (16,1.057756,0.574008,0.096116), (13,0.95373,0.029474,0.084782), (42,-1.164948,-0.7742,0.078442), (19,0.811068,0.323871,0.078177), (22,-0.659857,0.759794,0.073073), (17,0.925485,0.570557,0.068217), (30,-0.742054,-0.320013,0.062335), (38,-0.616053,-0.098321,0.059773), (44,0.780373,0.448802,0.050567), (29,0.608591,0.772367,0.048882), (25,0.645444,-0.378224,0.045837), (20,0.705451,0.241813,0.044182), (9,0.628304,0.033566,0.044028), (32,-0.242654,0.143798,0.043517)],
    50: [(32,0.214639,0.000358,0.329414), (28,0.317382,0.07659,0.120059), (3,0.314632,-0.018169,0.105797), (48,0.311025,0.033313,0.086404), (24,-0.058276,0.052737,0.080977), (36,0.241502,0.097979,0.077741), (1,0.24691,-0.119634,0.076467), (43,0.255133,-0.033372,0.063321), (18,0.259725,0.151465,0.060477), (0,0.200089,0.098641,0.046717), (40,-0.220753,0.05348,0.037871), (41,-0.141383,0.173666,0.032074), (31,-0.0492,0.132806,0.03178), (6,0.167129,0.079347,0.031701), (16,-0.192983,-0.008504,0.030953), (34,-0.039602,-0.008797,0.028684), (51,-0.143609,0.083541,0.025336), (4,-0.120502,0.281564,0.0215), (49,-0.046469,0.090153,0.020892), (8,-0.116747,0.050445,0.020604)],
    51: [(4,0.387091,-0.653138,0.180597), (41,0.36491,-0.252501,0.173922), (26,0.356655,-0.084979,0.134662), (31,0.105095,-0.127142,0.118036), (32,-0.138609,0.025047,0.111824), (39,0.275128,-0.053806,0.100986), (35,-0.332177,-0.022299,0.100892), (8,0.276738,0.057608,0.09424), (46,-0.291325,-0.201609,0.09279), (17,-0.369587,-0.241628,0.085675), (21,-0.292288,-0.461366,0.077243), (37,0.262894,0.068954,0.076931), (12,0.108229,-0.072205,0.073355), (20,-0.311197,-0.118586,0.067709), (29,-0.241148,-0.31996,0.060441), (19,-0.220236,-0.107427,0.045395), (23,-0.203274,0.122842,0.043679), (16,-0.238731,-0.15108,0.038557), (15,-0.245772,-0.255453,0.03508), (13,-0.216696,-0.028158,0.034469)],
    52: [(12,-0.79827,2.837303,0.226652), (33,-1.860129,2.981552,0.150132), (29,-1.293334,0.991282,0.098743), (10,-1.196879,3.761985,0.081492), (6,-1.195883,2.600302,0.075039), (32,-0.45897,2.725398,0.069637), (17,-1.376348,1.755677,0.067483), (39,0.943576,2.461667,0.067463), (23,-1.045846,3.332432,0.065669), (7,-0.97112,2.01154,0.063136), (35,-1.073991,2.567896,0.059902), (15,-1.254856,1.394884,0.051939), (31,-0.286693,2.798449,0.049888), (30,0.974789,3.016828,0.048114), (41,0.788755,2.060345,0.046152), (19,-0.896164,2.22928,0.04269), (21,-0.891291,1.227315,0.040794), (4,0.742928,1.344931,0.037783), (26,0.748027,2.425869,0.033644), (51,0.731596,2.560941,0.030399)],
}
    COL_MEANS = [165.276734, 165.697807, 164.269494, 161.908022, 157.794977, 158.87365, 167.157604, 164.761282, 162.867642, 161.968137, 164.440692, 164.984519, 164.059431, 159.707397, 164.242465, 158.635633, 161.642801, 161.940223, 163.704367, 167.123066, 162.606244, 158.492877, 159.238657, 162.699909, 159.814856, 163.456594, 166.514033, 161.632699, 162.469959, 160.302488, 163.649653, 162.929244, 161.449586, 164.486871, 156.835329, 166.940937, 164.148524, 161.762457, 158.656158, 163.66021, 160.016682, 167.338368, 163.037224, 162.3742, 163.470453, 163.121092, 158.804433, 158.951695, 161.063735, 159.782912, 165.067338, 159.735261, 165.188027]
    INDEX_COEFS = {
    'col_00': {'col_31': 0.2982, 'col_32': 0.2267, 'col_49': 0.2142, 'col_24': 0.1162, 'col_12': 0.086, 'col_34': 0.0435, 'col_52': 0.0037},
    'col_01': {'col_49': 0.2973, 'col_34': 0.2019, 'col_32': 0.1771, 'col_31': 0.1735, 'col_12': 0.158},
    'col_02': {'col_32': 0.2852, 'col_12': 0.2305, 'col_24': 0.1598, 'col_31': 0.1513, 'col_49': 0.0947, 'col_34': 0.0507, 'col_52': 0.0262},
    'col_03': {'col_49': 0.2697, 'col_32': 0.2261, 'col_12': 0.165, 'col_31': 0.1629, 'col_34': 0.1305, 'col_24': 0.0482},
    'col_04': {'col_49': 0.4044, 'col_34': 0.1861, 'col_32': 0.1225, 'col_12': 0.1094, 'col_31': 0.1006, 'col_24': 0.0923},
    'col_05': {'col_49': 0.29, 'col_12': 0.2476, 'col_52': 0.1806, 'col_32': 0.1448, 'col_31': 0.1412, 'col_34': 0.008},
    'col_06': {'col_12': 0.3848, 'col_31': 0.2594, 'col_32': 0.2371, 'col_24': 0.0625, 'col_49': 0.0478},
    'col_07': {'col_12': 0.3667, 'col_32': 0.2035, 'col_31': 0.1917, 'col_49': 0.1447, 'col_24': 0.0788, 'col_34': 0.0049},
    'col_08': {'col_12': 0.2211, 'col_49': 0.199, 'col_31': 0.1885, 'col_34': 0.1837, 'col_24': 0.1407, 'col_32': 0.0677},
    'col_09': {'col_24': 0.2902, 'col_32': 0.2425, 'col_31': 0.1771, 'col_49': 0.1402, 'col_52': 0.1339, 'col_34': 0.009, 'col_12': 0.0054},
    'col_10': {'col_49': 0.3891, 'col_34': 0.2248, 'col_12': 0.167, 'col_31': 0.1167, 'col_32': 0.1162},
    'col_11': {'col_49': 0.238, 'col_32': 0.2152, 'col_12': 0.1582, 'col_34': 0.148, 'col_24': 0.1187, 'col_31': 0.0962, 'col_52': 0.0202},
    'col_13': {'col_49': 0.2991, 'col_12': 0.2239, 'col_31': 0.1601, 'col_32': 0.1202, 'col_34': 0.1029, 'col_24': 0.0934},
    'col_14': {'col_12': 0.358, 'col_49': 0.2429, 'col_31': 0.1985, 'col_32': 0.1091, 'col_24': 0.0445, 'col_52': 0.0364},
    'col_15': {'col_49': 0.2744, 'col_12': 0.2381, 'col_32': 0.2147, 'col_34': 0.1121, 'col_31': 0.0816, 'col_52': 0.0716},
    'col_16': {'col_31': 0.2571, 'col_49': 0.2513, 'col_24': 0.1888, 'col_32': 0.1688, 'col_12': 0.0964, 'col_34': 0.0339},
    'col_17': {'col_49': 0.2366, 'col_32': 0.2011, 'col_12': 0.1723, 'col_24': 0.1586, 'col_31': 0.1546, 'col_34': 0.0694, 'col_52': 0.0073},
    'col_18': {'col_49': 0.2446, 'col_34': 0.1865, 'col_32': 0.1631, 'col_12': 0.1464, 'col_31': 0.1351, 'col_24': 0.1237},
    'col_19': {'col_49': 0.3245, 'col_32': 0.208, 'col_31': 0.194, 'col_12': 0.1641, 'col_34': 0.1102},
    'col_20': {'col_32': 0.2502, 'col_49': 0.1779, 'col_34': 0.1706, 'col_24': 0.1693, 'col_12': 0.1157, 'col_31': 0.0922, 'col_52': 0.025},
    'col_21': {'col_32': 0.2124, 'col_34': 0.1953, 'col_12': 0.1738, 'col_31': 0.1668, 'col_49': 0.1316, 'col_24': 0.1214},
    'col_22': {'col_49': 0.373, 'col_24': 0.1974, 'col_32': 0.1869, 'col_31': 0.1277, 'col_34': 0.0655, 'col_52': 0.0399, 'col_12': 0.0145},
    'col_23': {'col_32': 0.4033, 'col_24': 0.2019, 'col_12': 0.1443, 'col_31': 0.0972, 'col_52': 0.0887, 'col_34': 0.0623, 'col_49': 0.0048},
    'col_25': {'col_32': 0.2682, 'col_49': 0.2546, 'col_31': 0.2417, 'col_34': 0.1158, 'col_24': 0.0909, 'col_52': 0.0341},
    'col_26': {'col_49': 0.2693, 'col_31': 0.1938, 'col_34': 0.1771, 'col_24': 0.1552, 'col_32': 0.1119, 'col_12': 0.0496, 'col_52': 0.0404},
    'col_27': {'col_49': 0.3749, 'col_31': 0.1866, 'col_12': 0.1775, 'col_32': 0.1192, 'col_34': 0.0794, 'col_24': 0.0651},
    'col_28': {'col_49': 0.3615, 'col_12': 0.178, 'col_34': 0.1212, 'col_32': 0.1116, 'col_52': 0.0947, 'col_24': 0.0715, 'col_31': 0.0565},
    'col_29': {'col_12': 0.2496, 'col_49': 0.2268, 'col_32': 0.1854, 'col_31': 0.1193, 'col_24': 0.1067, 'col_34': 0.1048},
    'col_30': {'col_34': 0.2342, 'col_49': 0.2174, 'col_31': 0.164, 'col_24': 0.1409, 'col_32': 0.1179, 'col_12': 0.114, 'col_52': 0.0117},
    'col_33': {'col_49': 0.2641, 'col_12': 0.2294, 'col_31': 0.197, 'col_32': 0.1608, 'col_34': 0.133, 'col_24': 0.0163},
    'col_35': {'col_49': 0.4331, 'col_31': 0.1498, 'col_52': 0.1368, 'col_32': 0.1079, 'col_34': 0.0961, 'col_12': 0.0885},
    'col_36': {'col_49': 0.2698, 'col_32': 0.2323, 'col_12': 0.2249, 'col_31': 0.1382, 'col_34': 0.137},
    'col_37': {'col_24': 0.1881, 'col_31': 0.1748, 'col_12': 0.1658, 'col_32': 0.1655, 'col_49': 0.1548, 'col_52': 0.1166, 'col_34': 0.0258},
    'col_38': {'col_49': 0.3553, 'col_12': 0.1793, 'col_32': 0.1762, 'col_34': 0.1002, 'col_31': 0.0839, 'col_52': 0.0608, 'col_24': 0.0379},
    'col_39': {'col_12': 0.2455, 'col_31': 0.196, 'col_32': 0.1899, 'col_49': 0.1844, 'col_24': 0.1294, 'col_34': 0.0635},
    'col_40': {'col_49': 0.2821, 'col_31': 0.233, 'col_34': 0.1454, 'col_24': 0.1219, 'col_32': 0.0962, 'col_12': 0.0691, 'col_52': 0.0505},
    'col_41': {'col_49': 0.3123, 'col_32': 0.2347, 'col_31': 0.1579, 'col_34': 0.1513, 'col_12': 0.1366},
    'col_42': {'col_32': 0.2226, 'col_49': 0.206, 'col_31': 0.1972, 'col_34': 0.1212, 'col_52': 0.1124, 'col_24': 0.1087, 'col_12': 0.0249},
    'col_43': {'col_12': 0.2697, 'col_49': 0.1978, 'col_34': 0.1646, 'col_31': 0.1067, 'col_32': 0.1024, 'col_24': 0.0866, 'col_52': 0.0774},
    'col_44': {'col_49': 0.1978, 'col_12': 0.1892, 'col_31': 0.1846, 'col_24': 0.1775, 'col_32': 0.1599, 'col_52': 0.0533, 'col_34': 0.0331},
    'col_45': {'col_31': 0.231, 'col_34': 0.2205, 'col_12': 0.1656, 'col_24': 0.1414, 'col_32': 0.1357, 'col_49': 0.1081},
    'col_46': {'col_34': 0.3336, 'col_32': 0.2044, 'col_12': 0.1568, 'col_24': 0.1213, 'col_49': 0.0979, 'col_31': 0.0688, 'col_52': 0.0169},
    'col_47': {'col_49': 0.3221, 'col_12': 0.2235, 'col_31': 0.1433, 'col_34': 0.1079, 'col_32': 0.0884, 'col_24': 0.0656, 'col_52': 0.0398},
    'col_48': {'col_12': 0.239, 'col_32': 0.1943, 'col_31': 0.1902, 'col_49': 0.1457, 'col_24': 0.143, 'col_34': 0.052, 'col_52': 0.0288},
    'col_50': {'col_32': 0.4025, 'col_49': 0.1347, 'col_24': 0.1291, 'col_31': 0.1258, 'col_12': 0.1222, 'col_34': 0.0798},
    'col_51': {'col_31': 0.2271, 'col_49': 0.2068, 'col_32': 0.1827, 'col_12': 0.1795, 'col_24': 0.154, 'col_34': 0.0451},
}
    return {'COLS': COLS, 'COL_TO_IDX': COL_TO_IDX, 'COL_MEANS': COL_MEANS,
            'INDEX_COEFS': INDEX_COEFS, 'DEV_MODELS': DEV_MODELS,
            'INDEX_IDXS': [COL_TO_IDX[c] for c in INDEX_COEFS]}


def _load_data():
    """Return accumulated history arrays for temporal lookback.
    Rows are accumulated via _record_row() — works on held-out data."""
    if not hasattr(_load_data, '_rows'):
        _load_data._rows = []
        _load_data._cache = None
    if _load_data._cache is not None and len(_load_data._cache[0]) == len(_load_data._rows):
        return _load_data._cache
    if len(_load_data._rows) == 0:
        empty = np.zeros((0, 53), dtype=float)
        return (empty, np.ones((0, 53), dtype=bool))
    data = np.array(_load_data._rows, dtype=float)
    nm = np.isnan(data)
    _load_data._cache = (data, nm)
    return (data, nm)


def _record_row(prices):
    """Append current row to the streaming buffer."""
    if not hasattr(_load_data, '_rows'):
        _load_data._rows = []
        _load_data._cache = None
    cols = [f'col_{i:02d}' for i in range(53)]
    vals = [prices.get(c, np.nan) for c in cols]
    _load_data._rows.append(vals)
    _load_data._cache = None


def _parse_prices(prices, C):
    COLS = C['COLS']
    n = len(COLS)
    nan_cols, obs_cols = [], []
    row_values = np.zeros(n)
    for i, c in enumerate(COLS):
        v = prices.get(c, np.nan)
        if pd.isna(v):
            nan_cols.append(i)
        else:
            obs_cols.append(i)
            row_values[i] = v
    obs_set = set(obs_cols)
    market = row_values[obs_cols].mean() if obs_cols else np.mean(C['COL_MEANS'])
    obs_devs = {j: row_values[j] - market for j in obs_cols}
    t = None
    tv = prices.get('time', np.nan)
    if not pd.isna(tv):
        t = int(tv)
    return nan_cols, obs_cols, row_values, obs_set, market, obs_devs, t


def _reverse_inference(row_values, obs_set, C):
    solved = {}
    IDX = C['INDEX_COEFS']; CI = C['COL_TO_IDX']
    for _ in range(3):
        ns = 0
        for idx_name, coefs in IDX.items():
            idx_j = CI[idx_name]
            if idx_j not in obs_set and idx_j not in solved:
                continue
            idx_val = row_values[idx_j] if idx_j in obs_set else solved.get(idx_j)
            if idx_val is None:
                continue
            unknown, known_sum = [], 0.0
            for f, c in coefs.items():
                fj = CI[f]
                if fj in obs_set: known_sum += c * row_values[fj]
                elif fj in solved: known_sum += c * solved[fj]
                else: unknown.append((fj, c))
            if len(unknown) == 1:
                fj, c = unknown[0]
                if c >= 0.05 and fj not in solved:
                    val = (idx_val - known_sum) / c
                    if val > 0: solved[fj] = val; ns += 1
        for idx_name, coefs in IDX.items():
            idx_j = CI[idx_name]
            if idx_j in obs_set or idx_j in solved:
                continue
            all_k, val = True, 0.0
            for f, c in coefs.items():
                fj = CI[f]
                if fj in obs_set: val += c * row_values[fj]
                elif fj in solved: val += c * solved[fj]
                else: all_k = False; break
            if all_k and val > 0: solved[idx_j] = val; ns += 1
        if ns == 0: break
    return solved


def _predict_nan(ni, obs_set, obs_devs, market, row_values, t, C, data=None, nan_mask=None):
    COLS = C['COLS']; CI = C['COL_TO_IDX']; DM = C['DEV_MODELS']; CM = C['COL_MEANS']
    IDX = C['INDEX_COEFS']
    cname = COLS[ni]
    if cname in IDX:
        coefs = IDX[cname]
        fidxs = {CI[f]: c for f, c in coefs.items()}
        if all(fi in obs_set for fi in fidxs):
            price = sum(c * row_values[fi] for fi, c in fidxs.items())
            if price > 0: return price

    n_rows = data.shape[0] if data is not None else 0
    temporal_dev = None
    gap = 9999
    if t is not None and t > 0 and data is not None and t < n_rows:
        prev = np.where(~nan_mask[:t, ni])[0]
        if len(prev) > 0:
            gap = t - prev[-1]
            recent = prev[-3:]
            devs = []
            for rt in recent:
                obs_rt = np.where(~nan_mask[rt])[0]
                if len(obs_rt) >= 5:
                    devs.append(data[rt, ni] - data[rt, obs_rt].mean())
            if devs:
                temporal_dev = np.mean(devs)

    cross_dev = None
    if ni in DM:
        preds, weights = [], []
        for pj, slope, intercept, r2 in DM[ni]:
            if pj in obs_set:
                preds.append(slope * obs_devs[pj] + intercept)
                weights.append(r2)
        if len(preds) >= 2:
            w = np.array(weights); p = np.array(preds)
            cross_dev = np.dot(w, p) / w.sum()

    if temporal_dev is not None and cross_dev is not None:
        tw = max(0.77 ** gap, 0.05)
        return market + tw * temporal_dev + (1 - tw) * cross_dev
    if temporal_dev is not None:
        return market + temporal_dev
    if cross_dev is not None:
        return market + cross_dev
    return CM[ni]


def trading_problem_3(prices: pd.Series) -> pd.DataFrame:
    C = _get_constants()
    _record_row(prices)
    data, nan_mask = _load_data()
    nan_cols, obs_cols, rv, obs_set, mkt, obs_devs, _ = _parse_prices(prices, C)
    t = len(data) - 1
    if not nan_cols:
        return pd.DataFrame({'purchase_col': [C['COLS'][0]], 'qty': [100]})
    solved = _reverse_inference(rv, obs_set, C)
    preds = {}
    for ni in nan_cols:
        if ni in solved: preds[ni] = solved[ni]; continue
        preds[ni] = _predict_nan(ni, obs_set, obs_devs, mkt, rv, t, C, data, nan_mask)
    if not preds:
        return pd.DataFrame({'purchase_col': [C['COLS'][nan_cols[0]]], 'qty': [100]})
    best = min(preds, key=preds.get)
    return pd.DataFrame({'purchase_col': [C['COLS'][best]], 'qty': [100]})


def trading_problem_4(prices: pd.Series) -> pd.DataFrame:
    C = _get_constants()
    _record_row(prices)
    data, nan_mask = _load_data()
    nan_cols, obs_cols, rv, obs_set, mkt, obs_devs, _ = _parse_prices(prices, C)
    t = len(data) - 1
    if not nan_cols:
        return pd.DataFrame({'src_col': [], 'dest_col': [], 'qty': []}).astype({'src_col':str,'dest_col':str,'qty':int})
    solved = _reverse_inference(rv, obs_set, C)
    nan_prices = {}
    for ni in nan_cols:
        nan_prices[ni] = solved[ni] if ni in solved else _predict_nan(ni, obs_set, obs_devs, mkt, rv, t, C, data, nan_mask)
    dest_prices = {}
    for j in C['INDEX_IDXS']:
        if j in obs_set: dest_prices[j] = rv[j]
        elif j in nan_prices: dest_prices[j] = nan_prices[j]
    if not dest_prices:
        return pd.DataFrame({'src_col': [], 'dest_col': [], 'qty': []}).astype({'src_col':str,'dest_col':str,'qty':int})
    best_profit, best_src, best_dest = 0.0, None, None
    for dj, dp in dest_prices.items():
        for sj in nan_cols:
            if sj == dj: continue
            sp = nan_prices.get(sj, C['COL_MEANS'][sj])
            ep = dp - sp
            if ep > best_profit: best_profit = ep; best_src = sj; best_dest = dj
    if best_src is None or best_profit <= 0:
        return pd.DataFrame({'src_col': [], 'dest_col': [], 'qty': []}).astype({'src_col':str,'dest_col':str,'qty':int})
    return pd.DataFrame({'src_col': [C['COLS'][best_src]], 'dest_col': [C['COLS'][best_dest]], 'qty': [100]})


def trading_problem_5(prices: pd.Series) -> pd.DataFrame:
    C = _get_constants()
    _record_row(prices)
    data, nan_mask = _load_data()
    nan_cols, obs_cols, rv, obs_set, mkt, obs_devs, _ = _parse_prices(prices, C)
    t = len(data) - 1
    if not nan_cols:
        return pd.DataFrame({'order_col': [], 'px': [], 'qty': []})
    solved = _reverse_inference(rv, obs_set, C)
    preds = {}
    for ni in nan_cols:
        preds[ni] = solved[ni] if ni in solved else _predict_nan(ni, obs_set, obs_devs, mkt, rv, t, C, data, nan_mask)
    if not preds:
        return pd.DataFrame({'order_col': [], 'px': [], 'qty': []})
    best = min(preds, key=preds.get)
    bid = round(float(preds[best]) + 1.0, 2)
    return pd.DataFrame({'order_col': [C['COLS'][best]], 'px': [bid], 'qty': [100]})


## Performance Evaluation

Replay backtest with streaming buffer (reset between problems).

In [8]:
print("="*60)
print("BACKTESTING ALL TRADING FUNCTIONS (streaming)")
print("="*60)

_cols = [c for c in df.columns if c != 'time']
_nm = df[_cols].isna().to_numpy()
_imp = problem2_imputed_df[_cols].to_numpy(dtype=float)
_ci = {c: i for i, c in enumerate(_cols)}

# --- P3 --- (reset buffer)
if hasattr(_load_data, '_rows'):
    _load_data._rows = []
    _load_data._cache = None
p3_total, p3_oracle, p3_uniform, p3_days = 0.0, 0.0, 0.0, 0
for t in range(len(df)):
    row = df.iloc[t]
    ni = np.where(_nm[t])[0]
    if len(ni) == 0: continue
    res = trading_problem_3(row)
    ci = _ci[res['purchase_col'].values[0]]
    p3_days += 1
    p3_total += float(res['qty'].values[0]) * _imp[t, ci]
    p3_oracle += float(np.min(_imp[t, ni])) * 100.0
    p3_uniform += float(np.mean(_imp[t, ni])) * 100.0

print(f"\n--- Problem 3: Buy Cheap Flour ---")
print(f"  Rows traded:          {p3_days}")
print(f"  Excess over oracle:   {100*(p3_total/p3_oracle-1):.2f}%")
print(f"  Savings vs uniform:   {100*(1-p3_total/p3_uniform):.2f}%")

# --- P4 --- (reset buffer)
if hasattr(_load_data, '_rows'):
    _load_data._rows = []
    _load_data._cache = None
p4_profit, p4_traded, p4_loss, p4_skip = 0.0, 0, 0, 0
for t in range(len(df)):
    row = df.iloc[t]
    ni = np.where(_nm[t])[0]
    if len(ni) == 0: p4_skip += 1; continue
    res = trading_problem_4(row)
    if len(res) == 0: p4_skip += 1; continue
    p4_traded += 1
    sj = _ci[res['src_col'].values[0]]
    dj = _ci[res['dest_col'].values[0]]
    qty = int(res['qty'].values[0])
    pr = qty * (_imp[t, dj] - _imp[t, sj])
    p4_profit += pr
    if pr < 0: p4_loss += 1

print(f"\n--- Problem 4: Arbitrage ---")
print(f"  Days traded:          {p4_traded}")
print(f"  Days skipped:         {p4_skip}")
print(f"  Losses:               {p4_loss}")
print(f"  Total profit:         {p4_profit:,.0f}")

# --- P5 --- (reset buffer)
if hasattr(_load_data, '_rows'):
    _load_data._rows = []
    _load_data._cache = None
p5_score, p5_oracle, p5_traded, p5_fills, p5_orders = 0.0, 0.0, 0, 0, 0
for t in range(len(df)):
    row = df.iloc[t]
    ni = np.where(_nm[t])[0]
    if len(ni) == 0: continue
    p5_traded += 1
    res = trading_problem_5(row)
    med = float(np.median(_imp[t]))
    for _, tr in res.iterrows():
        cj = _ci[tr['order_col']]
        tp = _imp[t, cj]
        bp = float(tr['px'])
        qty = int(tr['qty'])
        p5_orders += 1
        if bp >= tp:
            p5_score += qty * (med - bp)
            p5_fills += 1
    p5_oracle += 100.0 * (med - float(np.min(_imp[t, ni])))

print(f"\n--- Problem 5: Limit Orders ---")
print(f"  Days traded:          {p5_traded}")
print(f"  Orders:               {p5_orders}")
print(f"  Fills:                {p5_fills} ({100*p5_fills/max(p5_orders,1):.1f}%)")
print(f"  Total score:          {p5_score:,.0f}")
print(f"  Oracle score:         {p5_oracle:,.0f}")
print(f"  Efficiency:           {100*p5_score/max(p5_oracle,1):.1f}%")

print(f"\n{'='*60}")
print("Expected from final_notebook_alt.ipynb:")
print("  P3 excess over oracle: 7.69%")
print("  P4 profit: 4,373,987 | losses: 481")
print("  P5 fills: 1405 (38.5%) | score: 740,593 | eff: 13.3%")
print("="*60)

BACKTESTING ALL TRADING FUNCTIONS (streaming)



--- Problem 3: Buy Cheap Flour ---
  Rows traded:          3650
  Excess over oracle:   7.69%
  Savings vs uniform:   2.33%



--- Problem 4: Arbitrage ---
  Days traded:          3650
  Days skipped:         0
  Losses:               481
  Total profit:         4,373,987



--- Problem 5: Limit Orders ---
  Days traded:          3650
  Orders:               3650
  Fills:                1405 (38.5%)
  Total score:          740,593
  Oracle score:         5,580,101
  Efficiency:           13.3%

Expected from final_notebook_alt.ipynb:
  P3 excess over oracle: 7.69%
  P4 profit: 4,373,987 | losses: 481
  P5 fills: 1405 (38.5%) | score: 740,593 | eff: 13.3%


## Summary

| Problem | Function | Format |
|---------|----------|--------|
| 1 | `solution()` | index_col, constituent_col, coef |
| 2 | `solve_problem_2(df)` + `submit_answer_2(...)` | filled CSV |
| 3 | `trading_problem_3(prices)` | purchase_col, qty |
| 4 | `trading_problem_4(prices)` | src_col, dest_col, qty |
| 5 | `trading_problem_5(prices)` | order_col, px, qty |

All functions self-contained, streaming temporal lookback, no global variables.